# QWEN_IM_F - Few-shot multimodal (Qwen3.5-27B / Gemma-4-31B) para EXIST 2026

Pipeline **few-shot puro** (sin fine-tuning) sobre los memes del reto, generando salidas **HARD** PyEvALL para las tres subtasks T2.1, T2.2 y T2.3.

Sirve como baseline LLM-grande comparable con `QWEN_VFINAL.ipynb` (XLM-R fine-tuneado con soft training).

## Workflow de uso

El notebook se controla con dos variables (celda 1.4):

- `BACKEND`  in {`"qwen35"`, `"gemma4"`} - selecciona la familia VLM.
- `STAGE`    in {`"sanity"`, `"submission"`} - selecciona el subset evaluado.

1. **`STAGE = "sanity"`** -> autoeval rapida sobre **800 memes de TRAIN**. Calcula `ICM`, `ICMNorm`, `FMeasure` para validar antes del run completo.
2. **`STAGE = "submission"`** -> run completo sobre los **1.053 memes de TEST oficial**. Empaqueta `exist2026_<team>/task2_X_hard_<team>_<run>.zip`.

Las caches se **namespacean** por backend (`cache_fewshot_qwen35/` vs `cache_fewshot_gemma4/`). Los **pools few-shot son compartidos**.

## Decisiones clave

- Modelos:
  - `qwen35` -> `Qwen/Qwen3.5-27B` (VLM unificado nativo: texto + imagen + video; ~27B; Feb 2026).
  - `gemma4` -> `google/gemma-4-31B-it` (fallback `google/gemma-3-27b-it`).
- Decoding **greedy** (`do_sample=False`).
- Pool few-shot **determinista** construido desde TRAIN.
- Solo archivos **HARD**.
- `test_case = "EXIST2025"` en todos los JSON.

## 1 · Setup

In [7]:
# === 1.1 · Detección Colab vs local ==========================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("IN_COLAB =", IN_COLAB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB = True


In [2]:
!pip install -U "git+https://github.com/huggingface/transformers.git"

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-a2kcfkq5
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-a2kcfkq5
  Resolved https://github.com/huggingface/transformers.git to commit f4757ac315558fb2f221ab52a3f73ab70a657970
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11873886 sha256=f29bb9cb4bbb92835c7674c914dcd8af004d0de2fe779a43efdd604ab6ffaf4a
  Stored in directory: /tmp/pip-ephem-wheel-cache-g5fjpyp4/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# === 1.2 · Dependencias ======================================================
# Qwen3-VL y Gemma-4 requieren transformers >= 4.57.
# Para Qwen-VL: qwen-vl-utils (sin extra 'decord', solo imagenes).
# Para Gemma: no hace falta lib extra (el processor maneja PIL directamente).
import sys, subprocess, importlib

PKGS = [
    "transformers>=4.57",
    "accelerate",
    "qwen-vl-utils",      # solo necesario si BACKEND='qwen3vl', inofensivo si no
    "pillow",
    "tqdm",
    "scikit-learn",
]
for p in PKGS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

# bitsandbytes solo si se va a quantizar (GPUs <24GB qwen, <48GB gemma)
try:
    import bitsandbytes  # noqa
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
    except Exception as e:
        print("[warn] bitsandbytes no se pudo instalar:", e)

# Verificacion inmediata de qwen_vl_utils (solo si vas a usar BACKEND='qwen3vl')
try:
    importlib.import_module("qwen_vl_utils")
    print("OK: qwen_vl_utils importado correctamente")
except ImportError:
    print("[warn] qwen_vl_utils no se importa, reinstalando...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                            "--no-cache-dir", "qwen-vl-utils"])
    importlib.invalidate_caches()
    try:
        import qwen_vl_utils  # noqa
        print("OK: qwen_vl_utils tras reinstalar")
    except ImportError:
        print("[info] qwen_vl_utils sigue sin disponible (OK si vas a usar BACKEND='gemma4')")


KeyboardInterrupt: 

In [8]:
# === 1.3 · Imports + reproducibilidad ========================================
import json, os, random, gc, warnings, re, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")

DEVICE: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM: 102.0 GB


In [9]:
# === 1.4 · Rutas del proyecto + workflow ======================================
# BACKEND selecciona la familia VLM:
#   "qwen35"  -> Qwen/Qwen3.5-27B        (VLM unificado nativo, ~27B, Feb 2026)
#   "gemma4"  -> google/gemma-4-31B-it      (fallback google/gemma-3-27b-it)
#
# STAGE controla el subset evaluado:
#   "sanity"     → autoeval rápida sobre SANITY_N memes de TRAIN (ICM/ICMNorm/FMeasure)
#   "submission" → run completo sobre los 1.053 memes de TEST oficial
#
# Cachés namespaceadas por backend para alternar entre modelos sin colisión.
BACKEND  = "gemma4"        # "qwen35" | "gemma4"
STAGE    = "submission"     # "sanity"  | "submission"
SANITY_N = 800

assert BACKEND in ("qwen35", "gemma4"), f"BACKEND desconocido: {BACKEND}"

if STAGE == "sanity":
    RUN_MODE       = "train_eval"
    MAX_EVAL_MEMES = SANITY_N
elif STAGE == "submission":
    RUN_MODE       = "test"
    MAX_EVAL_MEMES = None
else:
    raise ValueError(f"STAGE desconocido: {STAGE}")

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/EXIST_2026")
else:
    PROJECT_ROOT = Path(r"c:/Users/juana/OneDrive/Escritorio/TFG/EXIST_2026")

DATASET_DIR = PROJECT_ROOT / "EXIST 2026 Dataset V0.1"

TRAIN_JSON_CANDIDATES = [
    DATASET_DIR / "EXIST 2026 Memes Dataset" / "training" / "EXIST2026_training_memes.json",
    DATASET_DIR / "EXIST2026_training_memes.json",
]
TRAIN_MEME_DIR_CANDIDATES = [
    DATASET_DIR / "EXIST 2026 Memes Dataset" / "training" / "memes",
    DATASET_DIR / "memes",
]
TEST_JSON_CANDIDATES = [
    DATASET_DIR / "Dataset - Evall" / "EXIST 2026 Memes Dataset" / "test" / "EXIST2026_test_clean.json",
    PROJECT_ROOT / "Dataset - Evall" / "EXIST 2026 Memes Dataset" / "test" / "EXIST2026_test_clean.json",
]
TEST_MEME_DIR_CANDIDATES = [
    DATASET_DIR / "Dataset - Evall" / "EXIST 2026 Memes Dataset" / "test" / "memes",
    PROJECT_ROOT / "Dataset - Evall" / "EXIST 2026 Memes Dataset" / "test" / "memes",
]

def _first_existing(cands):
    for c in cands:
        if c.exists(): return c
    return None

TRAIN_JSON     = _first_existing(TRAIN_JSON_CANDIDATES)
TRAIN_MEME_DIR = _first_existing(TRAIN_MEME_DIR_CANDIDATES)
TEST_JSON      = _first_existing(TEST_JSON_CANDIDATES)
TEST_MEME_DIR  = _first_existing(TEST_MEME_DIR_CANDIDATES)

assert TRAIN_JSON is not None,     f"No se encontró JSON train: {TRAIN_JSON_CANDIDATES}"
assert TRAIN_MEME_DIR is not None, f"No se encontró dir train memes: {TRAIN_MEME_DIR_CANDIDATES}"
if RUN_MODE == "test":
    assert TEST_JSON is not None,     f"No se encontró JSON test: {TEST_JSON_CANDIDATES}"
    assert TEST_MEME_DIR is not None, f"No se encontró dir test memes: {TEST_MEME_DIR_CANDIDATES}"

# Cache namespaceada por backend (predictions/checkpoints/submission son del modelo).
# POOLS_DIR es COMPARTIDA porque los pools dependen solo del dataset, no del modelo,
# y mantenemos la ruta histórica `cache_qwen_fewshot/few_shot_pools` para no
# regenerarlos cuando ya existen.
CACHE_DIR = PROJECT_ROOT / f"cache_fewshot_{BACKEND}"
POOLS_DIR = PROJECT_ROOT / "cache_qwen_fewshot" / "few_shot_pools"
PRED_DIR  = CACHE_DIR / "predictions"
CKPT_DIR  = CACHE_DIR / "checkpoints"
METR_DIR  = CACHE_DIR / "metrics"
LOGS_DIR  = CACHE_DIR / "logs"
PYE_WORK  = CACHE_DIR / "pyevall_work"
SUBM_DIR  = CACHE_DIR / "submission"
for d in [CACHE_DIR, POOLS_DIR, PRED_DIR, CKPT_DIR, METR_DIR, LOGS_DIR, PYE_WORK, SUBM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"BACKEND        : {BACKEND}")
print(f"STAGE          : {STAGE}")
print(f"RUN_MODE       : {RUN_MODE}")
print(f"MAX_EVAL_MEMES : {MAX_EVAL_MEMES}")
print(f"TRAIN_JSON     : {TRAIN_JSON}")
print(f"TRAIN_MEME_DIR : {TRAIN_MEME_DIR} | count={len(list(TRAIN_MEME_DIR.glob('*.jpeg')))}")
if RUN_MODE == "test":
    print(f"TEST_JSON      : {TEST_JSON}")
    print(f"TEST_MEME_DIR  : {TEST_MEME_DIR} | count={len(list(TEST_MEME_DIR.glob('*.jpeg')))}")
print(f"CACHE_DIR      : {CACHE_DIR}")
print(f"POOLS_DIR      : {POOLS_DIR}  (compartido entre backends)")


BACKEND        : gemma4
STAGE          : submission
RUN_MODE       : test
MAX_EVAL_MEMES : None
TRAIN_JSON     : /content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/EXIST2026_training_memes.json
TRAIN_MEME_DIR : /content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/memes | count=3679
TEST_JSON      : /content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/Dataset - Evall/EXIST 2026 Memes Dataset/test/EXIST2026_test_clean.json
TEST_MEME_DIR  : /content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/Dataset - Evall/EXIST 2026 Memes Dataset/test/memes | count=973
CACHE_DIR      : /content/drive/MyDrive/EXIST_2026/cache_fewshot_gemma4
POOLS_DIR      : /content/drive/MyDrive/EXIST_2026/cache_qwen_fewshot/few_shot_pools  (compartido entre backends)


In [5]:
# === 1.5 · Carga del modelo VLM ==============================================
# Dispatch por BACKEND:
#   qwen35  -> Qwen/Qwen3.5-27B          (VLM unificado nativo Feb 2026)
#   gemma4  -> google/gemma-4-31B-it     (multimodal denso 31B, NO tiene fallback)
#
# Gemma-4 NO esta expuesta como Gemma4ForConditionalGeneration; se carga con
# AutoModelForMultimodalLM (mismo patron que QWEN_VID_FSHOT.ipynb).
#
# Requiere transformers >= 4.57. Si Gemma esta gated, ejecutar antes:
#     from huggingface_hub import login; login()
# y aceptar la licencia en https://huggingface.co/google/gemma-4-31B-it
#
# Quantizacion 4-bit automatica si VRAM < umbral (24GB qwen, 48GB gemma).
import transformers
from transformers import AutoProcessor

print("transformers version:", transformers.__version__)

if BACKEND == "qwen35":
    # Qwen3.5-27B es VLM unificado. La clase exacta puede variar entre versiones
    # de transformers; probamos varias y caemos a AutoModelForImageTextToText.
    MODEL_CANDIDATES = [
        ("Qwen/Qwen3.5-27B", "Qwen3_5ForConditionalGeneration"),
        ("Qwen/Qwen3.5-27B", "Qwen3_5VLForConditionalGeneration"),
        ("Qwen/Qwen3.5-27B", "AutoModelForImageTextToText"),
        ("Qwen/Qwen3.5-27B", "AutoModelForCausalLM"),
    ]
    VRAM_THRESHOLD = 48 * 1e9
elif BACKEND == "gemma4":
    # Gemma-4 usa la clase Auto multimodal. Si tu version de transformers aun
    # no expone `model_type='gemma4'` (typicamente <5.x final), cae al
    # fallback Gemma-3-27B-IT, soportado desde 4.57 y tambien multimodal.
    # Para forzar Gemma-4 instala transformers desde git:
    #   pip install -U "git+https://github.com/huggingface/transformers.git"
    MODEL_CANDIDATES = [
        ("google/gemma-4-31B-it", "AutoModelForMultimodalLM")
    ]
    VRAM_THRESHOLD = 48 * 1e9
else:
    raise ValueError(f"BACKEND desconocido: {BACKEND}")

USE_4BIT = (torch.cuda.is_available() and
            torch.cuda.get_device_properties(0).total_memory < VRAM_THRESHOLD)
print(f"USE_4BIT = {USE_4BIT}  (threshold {VRAM_THRESHOLD/1e9:.0f} GB)")

def _resolve_model_class(class_name):
    """Localiza la clase de modelo dentro de transformers. Lanza ImportError
    explicito si la version instalada no la expone."""
    cls = getattr(transformers, class_name, None)
    if cls is None:
        raise ImportError(
            f"{class_name} no esta disponible en transformers {transformers.__version__}. "
            f"Actualiza a transformers>=4.57 (reinicia el kernel tras el upgrade)."
        )
    return cls

model = None; processor = None; MODEL_ID = None; MODEL_CLASS_NAME = None
for mid, cls_name in MODEL_CANDIDATES:
    try:
        cls = _resolve_model_class(cls_name)
        kwargs = dict(device_map="auto")
        if USE_4BIT:
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
            )
        else:
            # Nota: en transformers >=4.57 el kwarg es `dtype`, no `torch_dtype`.
            kwargs["dtype"] = torch.bfloat16
        model = cls.from_pretrained(mid, **kwargs)
        processor = AutoProcessor.from_pretrained(mid)
        MODEL_ID = mid; MODEL_CLASS_NAME = cls.__name__
        print(f"OK Modelo cargado: {mid}  (cls={MODEL_CLASS_NAME}, 4bit={USE_4BIT})")
        break
    except Exception as e:
        print(f"[fail] {mid}: {type(e).__name__}: {e}")

assert model is not None, (
    f"Ningun modelo de la familia {BACKEND} pudo cargarse.\n"
    f"  - transformers={transformers.__version__} (necesario >=4.57)\n"
    f"  - Si Qwen3.5 esta gated: acepta licencia en https://huggingface.co/Qwen/Qwen3.5-27B\n"
    f"  - Si Gemma esta gated: `from huggingface_hub import login; login()` "
    f"y acepta la licencia en https://huggingface.co/google/gemma-4-31B-it"
)
model.eval()
print("Modelo en eval mode.")


transformers version: 5.8.0.dev0
USE_4BIT = False  (threshold 48 GB)


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

...

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

OK Modelo cargado: google/gemma-4-31B-it  (cls=AutoModelForMultimodalLM, 4bit=False)
Modelo en eval mode.


In [14]:
# Mira 10 respuestas RAW del modelo + cuenta fallbacks del parser
from pathlib import Path
import json

print(f"BACKEND = {BACKEND}  MODEL_ID = {MODEL_ID}\n")

# 1) Respuestas raw a 10 memes
eval_pool = df_train if RUN_MODE == "train_eval" else df_test
sample = eval_pool[~eval_pool["id_EXIST"].isin(POOL_IDS)].head(10)
print("--- 10 respuestas raw del modelo en T2.1 ---")
for _, row in sample.iterrows():
    pred, raw = predict_meme(row["id_EXIST"], row["img_path"], row["text_ocr"],
                              "t21", POOLS["t21"], return_raw=True)
    print(f"  id={row['id_EXIST']}  PRED={pred:3s}  RAW={raw!r}")

# 2) Cuenta de fallbacks del parser hasta ahora
fb_log = LOGS_DIR / "parse_fallbacks.log"
if fb_log.exists():
    lines = fb_log.read_text(encoding="utf-8").splitlines()
    print(f"\n--- parse_fallbacks.log: {len(lines)} fallbacks totales ---")
    for line in lines[:5]:
        d = json.loads(line)
        print(f"  task={d['task']}  id={d['id']}  resp={d['resp']!r}")
else:
    print("\n(no hay parse_fallbacks.log -> parser nunca cayó al fallback)")

# 3) Verifica que el pool tiene YES y NO
print("\n--- Pool T2.1 ---")
for ex in POOLS["t21"]:
    print(f"  id={ex['id']}  label={ex['label']}  cons={ex['consensus']:.2f}")


BACKEND = gemma4  MODEL_ID = google/gemma-4-31B-it

--- 10 respuestas raw del modelo en T2.1 ---


NameError: name 'predict_meme' is not defined

## 2 · Carga de datos + hard labels

Reutiliza los helpers de `QWEN_VFINAL.ipynb` (`hard_t21`, `hard_t22`, `hard_t23_multihot`) con los **umbrales oficiales** del lab guidelines V0.5 pág. 17:

- T2.1: `>3` votos.
- T2.2: `>2` votos (tras `"-" → "NO"`).
- T2.3: cada categoría con `>1` voto.

In [11]:
# === 2.1 · Hard label helpers + carga de JSON (train + test unificados) =====
SEXISM_CATS = [
    "IDEOLOGICAL-INEQUALITY", "STEREOTYPING-DOMINANCE", "OBJECTIFICATION",
    "SEXUAL-VIOLENCE", "MISOGYNY-NON-SEXUAL-VIOLENCE",
]
T21_INT_TO_STR = {0: "NO", 1: "YES"}
T22_INT_TO_STR = {0: "NO", 1: "DIRECT", 2: "JUDGEMENTAL"}
T22_STR_TO_INT = {v: k for k, v in T22_INT_TO_STR.items()}

def hard_t21(labs):
    yes = sum(1 for v in labs if v == "YES")
    no  = sum(1 for v in labs if v == "NO")
    if yes > 3: return 1
    if no  > 3: return 0
    return None

def hard_t22(labs):
    clean = [l for l in labs if l != "UNKNOWN"]
    if not clean: return None
    mapped = ["NO" if l == "-" else l for l in clean]
    c = Counter(mapped)
    top, votes = c.most_common(1)[0]
    if votes <= 2: return None
    return T22_STR_TO_INT[top]

def hard_t23_multihot(arrs):
    c = Counter()
    for arr in arrs:
        for l in arr:
            if l != "UNKNOWN": c[l] += 1
    vec = np.zeros(len(SEXISM_CATS), dtype=np.float32)
    for i, cat in enumerate(SEXISM_CATS):
        if c.get(cat, 0) > 1: vec[i] = 1.0
    return vec

def _build_rows(raw_dict, meme_dir, split):
    out = []
    for mid, m in raw_dict.items():
        img_name = m.get("meme") or f"{mid}.jpeg"
        img_path = meme_dir / img_name
        labs21 = m.get("labels_task2_1") or []
        labs22 = m.get("labels_task2_2") or []
        arrs23 = m.get("labels_task2_3") or []
        out.append({
            "id_EXIST" : str(mid),
            "split"    : split,                           # "train" | "test"
            "lang"     : m.get("lang", "en"),
            "text_ocr" : (m.get("text") or "").strip(),
            "img_path" : str(img_path),
            "img_ok"   : img_path.exists(),
            "t21_hard" : hard_t21(labs21) if labs21 else None,
            "t22_hard" : hard_t22(labs22) if labs22 else None,
            "t23_hard" : hard_t23_multihot(arrs23) if arrs23 else
                         np.zeros(len(SEXISM_CATS), dtype=np.float32),
        })
    return out

# ---- Carga TRAIN (siempre necesario para construir el few-shot pool) -------
with open(TRAIN_JSON, "r", encoding="utf-8", errors="replace") as f:
    raw_train = json.load(f)
rows_train = _build_rows(raw_train, TRAIN_MEME_DIR, "train")

# ---- Carga TEST (sólo si RUN_MODE=="test") ----------------------------------
raw_test, rows_test = {}, []
if RUN_MODE == "test":
    with open(TEST_JSON, "r", encoding="utf-8", errors="replace") as f:
        raw_test = json.load(f)
    rows_test = _build_rows(raw_test, TEST_MEME_DIR, "test")

df  = pd.DataFrame(rows_train + rows_test)
# `raw` unificado para lookups por id (build_pyevall_gold_hard, etc.)
raw = {**raw_train, **raw_test}

# Aliases convenientes
df_train = df[df["split"] == "train"].reset_index(drop=True)
df_test  = df[df["split"] == "test"].reset_index(drop=True)

print(f"Memes TRAIN cargados      : {len(df_train):,}")
print(f"  imágenes encontradas    : {df_train['img_ok'].sum():,}")
print(f"  t21_hard válida (>3)    : {df_train['t21_hard'].notna().sum():,}")
print(f"  t22_hard válida (>2)    : {df_train['t22_hard'].notna().sum():,}")
if RUN_MODE == "test":
    print(f"\nMemes TEST cargados       : {len(df_test):,}")
    print(f"  imágenes encontradas    : {df_test['img_ok'].sum():,}")
    print(f"  (test sin labels: subtask cols quedan en None / vec ceros)")

assert df_train["img_ok"].all(), "Hay memes train sin imagen; revisa TRAIN_MEME_DIR"
if RUN_MODE == "test":
    assert df_test["img_ok"].all(), "Hay memes test sin imagen; revisa TEST_MEME_DIR"

Memes TRAIN cargados      : 3,984
  imágenes encontradas    : 3,984
  t21_hard válida (>3)    : 3,370
  t22_hard válida (>2)    : 3,763

Memes TEST cargados       : 1,053
  imágenes encontradas    : 1,053
  (test sin labels: subtask cols quedan en None / vec ceros)


## 3 · Selección del few-shot pool (anti-leakage, alta consenso)

Modo **holdout**: separamos un pool fijo de IDs que NUNCA aparecen en el eval. Selección estratificada por clase, ordenando por consenso entre anotadores. Determinista (mismo seed → mismo pool).

- T2.1: 4 ejemplos (2 YES + 2 NO).
- T2.2: 6 ejemplos (2 NO + 2 DIRECT + 2 JUDGEMENTAL).
- T2.3: 12 ejemplos (2 NO + 2 por cada categoría sexista, prefiriendo memes single-label).

In [13]:
# === 3.1 - Builder de few-shot pools ==========================================
# IMPORTANTE: el pool se construye SIEMPRE desde TRAIN (los unicos memes con
# labels). Asi evitamos contaminacion con test y mantenemos el few-shot pool
# estable independientemente de RUN_MODE.
#
# DISENO HIERARCHICAL-CONDITIONAL (post-patch):
#   - T2.1 ve YES/NO -> pool con 2 YES + 2 NO.
#   - T2.2 SOLO recibe memes ya marcados como sexistas por T2.1 -> pool con
#     SOLO ejemplos DIRECT y JUDGEMENTAL (3+3 = 6 totales). Sin NO.
#   - T2.3 idem -> pool con SOLO ejemplos sexistas, single-label por categoria
#     cuando es posible (2 por cada una de las 5 cats = 10 totales). Sin NO.
#
# Esto elimina la contradiccion entre system prompt (que prohibe NO en
# T2.2/T2.3) y few-shot examples.
#
# NOTA: pandas convierte los None de columnas mixtas a NaN (float), y
# `NaN is not None`, asi que comparamos con `pd.isna(...)`.
N_PER_CLASS_T21 = 2

def _consensus_t21(labs):
    valid = [v for v in labs if v != "UNKNOWN"]
    if not valid: return 0.0
    return Counter(valid).most_common(1)[0][1] / len(valid)

def _consensus_t22(labs):
    valid = [l for l in labs if l != "UNKNOWN"]
    if not valid: return 0.0
    mapped = ["NO" if l == "-" else l for l in valid]
    return Counter(mapped).most_common(1)[0][1] / len(mapped)

def select_few_shot_pool_t21(df_pool, raw_pool, n_per=N_PER_CLASS_T21):
    cands = []
    for _, row in df_pool.iterrows():
        if pd.isna(row["t21_hard"]) or not row["img_ok"]: continue
        cons = _consensus_t21(raw_pool[row["id_EXIST"]].get("labels_task2_1") or [])
        cands.append((row["id_EXIST"], int(row["t21_hard"]), cons))
    cands.sort(key=lambda x: (-x[2], x[0]))
    pool = []
    for cls in (1, 0):
        chosen = [c for c in cands if c[1] == cls][:n_per]
        pool.extend(chosen)
    return [{"id": mid, "label": T21_INT_TO_STR[lbl], "consensus": float(cons)}
            for (mid, lbl, cons) in pool]

def select_few_shot_pool_t22(df_pool, raw_pool, n_per=3):
    """Pool SOLO con DIRECT y JUDGEMENTAL (sin NO).

    n_per=3 -> 6 ejemplos totales (3 DIRECT + 3 JUDGEMENTAL). Mas densidad
    de senal discriminativa que la version anterior 2+2+2 con NO incluido.
    """
    cands = []
    for _, row in df_pool.iterrows():
        if pd.isna(row["t22_hard"]) or not row["img_ok"]: continue
        cls = int(row["t22_hard"])
        if cls == 0: continue   # excluimos NO (no aplica en flujo condicional)
        cons = _consensus_t22(raw_pool[row["id_EXIST"]].get("labels_task2_2") or [])
        cands.append((row["id_EXIST"], cls, cons))
    cands.sort(key=lambda x: (-x[2], x[0]))
    pool = []
    for cls in (1, 2):  # DIRECT, JUDGEMENTAL
        chosen = [c for c in cands if c[1] == cls][:n_per]
        pool.extend(chosen)
    return [{"id": mid, "label": T22_INT_TO_STR[lbl], "consensus": float(cons)}
            for (mid, lbl, cons) in pool]

def select_few_shot_pool_t23(df_pool, raw_pool, n_per=2):
    """Pool SOLO con ejemplos sexistas (sin NO), prefiriendo single-label
    por categoria. Score = single*0.6 + cons*0.4.

    n_per=2 -> 10 ejemplos totales (2 por cada una de las 5 cats sexistas).
    """
    pool, used = [], set()
    for ci, cat in enumerate(SEXISM_CATS):
        cands = []
        for _, row in df_pool.iterrows():
            if row["id_EXIST"] in used or not row["img_ok"]: continue
            multi = row["t23_hard"]
            if multi is None or multi[ci] < 0.5: continue
            arrs = raw_pool[row["id_EXIST"]].get("labels_task2_3") or []
            valid = [a for a in arrs if "UNKNOWN" not in a]
            if not valid: continue
            n_single = sum(1 for a in valid if cat in a and len(a) == 1)
            n_with   = sum(1 for a in valid if cat in a)
            single   = n_single / len(valid)
            cons     = n_with   / len(valid)
            score    = single * 0.6 + cons * 0.4
            cands.append((row["id_EXIST"], score, cons))
        cands.sort(key=lambda x: (-x[1], x[0]))
        for mid, score, cons in cands[:n_per]:
            multi = df_pool.loc[df_pool["id_EXIST"]==mid, "t23_hard"].iloc[0]
            picks = [SEXISM_CATS[k] for k in range(5) if multi[k] > 0.5]
            pool.append({"id": mid, "labels": picks, "consensus": float(cons)})
            used.add(mid)
    return pool

POOL_PATHS = {
    "t21": POOLS_DIR / "few_shot_pool_t21.json",
    "t22": POOLS_DIR / "few_shot_pool_t22.json",
    "t23": POOLS_DIR / "few_shot_pool_t23.json",
}

def _pool_is_stale(task, cached):
    """Detecta caches generados con la version anterior (que incluia NO en
    t22/t23). Si encuentra entradas con NO -> obsoleto, regenerar."""
    if task == "t22":
        return any(ex.get("label") == "NO" for ex in cached)
    if task == "t23":
        return any(ex.get("labels") == ["NO"] for ex in cached)
    return False  # t21 siempre incluye NO, ese es valido

POOLS = {}
for task, builder in [("t21", select_few_shot_pool_t21),
                       ("t22", select_few_shot_pool_t22),
                       ("t23", select_few_shot_pool_t23)]:
    p = POOL_PATHS[task]
    if p.exists():
        cached = json.load(open(p, "r", encoding="utf-8"))
        if _pool_is_stale(task, cached):
            print(f"[{task}] pool cacheado obsoleto (incluye NO con diseno antiguo). Regenerando...")
            POOLS[task] = builder(df_train, raw_train)
            with open(p, "w", encoding="utf-8") as f:
                json.dump(POOLS[task], f, ensure_ascii=False, indent=2)
            print(f"[{task}] pool regenerado ({len(POOLS[task])} ej.) -> {p.name}")
        else:
            POOLS[task] = cached
            print(f"[{task}] pool cargado de cache ({len(POOLS[task])} ej.) - {p.name}")
    else:
        POOLS[task] = builder(df_train, raw_train)
        with open(p, "w", encoding="utf-8") as f:
            json.dump(POOLS[task], f, ensure_ascii=False, indent=2)
        print(f"[{task}] pool generado ({len(POOLS[task])} ej.) -> {p.name}")

# Conjunto de IDs en cualquier pool: se EXCLUYEN del eval (anti-leakage).
POOL_IDS = set()
for task in ["t21", "t22", "t23"]:
    for ex in POOLS[task]: POOL_IDS.add(str(ex["id"]))
print(f"Total IDs unicos en pools: {len(POOL_IDS)}")

# Resumen visual de los pools
for task in ["t21", "t22", "t23"]:
    print()
    print(f"  --- pool {task} ---")
    for ex in POOLS[task]:
        lbl = ex.get("label") or ex.get("labels")
        print(f"    id={ex['id']:>8}  cons={ex['consensus']:.2f}  label={lbl}")


[t21] pool cargado de cache (4 ej.) - few_shot_pool_t21.json
[t22] pool cargado de cache (6 ej.) - few_shot_pool_t22.json
[t23] pool cargado de cache (10 ej.) - few_shot_pool_t23.json
Total IDs unicos en pools: 19

  --- pool t21 ---
    id=  110001  cons=1.00  label=YES
    id=  110002  cons=1.00  label=YES
    id=  110018  cons=1.00  label=NO
    id=  110026  cons=1.00  label=NO

  --- pool t22 ---
    id=  110001  cons=1.00  label=DIRECT
    id=  110006  cons=1.00  label=DIRECT
    id=  110007  cons=1.00  label=DIRECT
    id=  110464  cons=1.00  label=JUDGEMENTAL
    id=  110516  cons=1.00  label=JUDGEMENTAL
    id=  210587  cons=1.00  label=JUDGEMENTAL

  --- pool t23 ---
    id=  110023  cons=1.00  label=['IDEOLOGICAL-INEQUALITY']
    id=  110286  cons=1.00  label=['IDEOLOGICAL-INEQUALITY']
    id=  110068  cons=1.00  label=['STEREOTYPING-DOMINANCE']
    id=  110400  cons=1.00  label=['STEREOTYPING-DOMINANCE']
    id=  110490  cons=1.00  label=['OBJECTIFICATION']
    id=  210259  

## 4 · Plantillas de prompt por subtask

Definiciones literales del **Lab Guidelines V0.5** (págs. 2–3) para minimizar ambigüedad.

Estructura: `system` con definición rigurosa + turnos `user / assistant` con N ejemplos few-shot intercalados + `user` final con el meme query.

In [16]:
# === 4.1 · Construcción de mensajes (chat-template Qwen-VL) ==================
# Cada elemento del pool se materializa como un par (user con imagen+texto,
# assistant con la respuesta). El meme query se añade al final como user sin
# assistant; `apply_chat_template(... add_generation_prompt=True)` cierra el
# mensaje para que el modelo genere a continuación.

SYSTEM_T21 = """You are an expert annotator specialized in identifying sexism in bilingual (English/Spanish) social-media memes.
 
# TASK
Decide whether a meme is SEXIST (YES) or NOT SEXIST (NO).
 
# WHAT A MEME IS
A meme is a multimodal artifact: meaning emerges from the COMBINATION of the image and the overlaid text, not from either in isolation. Read the OCR text, look at the visual content (people, gestures, setting, juxtapositions), and decide based on how they interact. A bland image can carry a sexist joke through the text; a neutral text can frame a degrading image.
 
# DEFINITION 
A meme is SEXIST if it contains sexist expressions or behaviours — that is, if it is sexist itself, describes a sexist situation, OR criticizes a sexist behaviour. The last clause matters: a meme that condemns sexism still deals with sexism, so it is YES.
 
# MARK YES WHEN ANY OF THESE APPLY
 
1. Overt sexism. Slurs, insults, harassment, or dehumanizing depictions targeting women or LGBTQ+ people on the basis of gender or gender identity.
 
2. Gender stereotypes used as the joke's premise. "Women drivers", "send her back to the kitchen", "men can't show emotions", "her place is at home". The humor only lands because of an essentialist claim about a gender.
 
3. Benevolent sexism. Apparently positive framings that tie women to traditional roles or restrictive virtues: "a real woman cooks for her man", "good girls don't…", "women are sacred mothers". This is one of the most common false negatives for content models — DO NOT skip it because it sounds nice.
 
4. Anti-feminism / men-as-victims. Discrediting feminism, claiming gender equality is already achieved, framing men as oppressed by women or by the law, mocking #MeToo, etc. Includes "feminazi" framings and similar.
 
5. Sexualization or objectification. Reducing a woman to her body, treating her appearance as her main worth, hypersexualized framings, "rate her" memes, body-shaming.
 
6. Sexual violence framing. Jokes about rape, harassment, coercion, or stalking — even when "humorous".
 
7. Counter-speech / critique of sexism. Feminist memes denouncing sexist behaviour, parodies of misogynists, screenshots of sexist comments held up for ridicule. Per the EXIST definition this is YES because the meme deals with sexism. Do NOT confuse this rule with letting through actually sexist content; the meme must clearly condemn sexism for this clause to apply.
 
8. Ironic / sarcastic memes whose punchline reinforces a stereotype. Irony does not flip the label here. "Yeah, women love being interrupted /s" is still sexism
 
# MARK NO WHEN
 
9. The meme is unrelated to gender — programming jokes, sports, animals, pop culture references where gender is not the hinge.
 
10. The meme depicts a woman but the joke is NOT about her gender. A meme of a female politician where the punchline is a policy position; a meme of a female celebrity where the joke is about her acting; a meme of a woman scientist celebrated for her work. If you can swap the gender and the joke still lands identically, it is almost certainly NO.
 
11. The meme references women neutrally without invoking inferiority, hierarchy, sexualization, restriction, or essentialist traits.
 
# DECISION RULES FOR HARD CASES
 
- ARBITER QUESTION: "Does this joke or message require a specific gender to land?" If yes, lean YES. If swapping the gender breaks nothing, lean NO.
- TEXT vs IMAGE CONFLICT: trust the explicit signal that carries the meme's communicative intent. If the OCR is overtly sexist and the image is benign stock photo, it is YES. If the image is degrading and the text is neutral, look at what the meme as a whole communicates.
- NO TEXT EXTRACTED: judge from the image alone using the same rules.
- BILINGUAL: text may be English or Spanish; both are in scope. Apply the same criteria.
- WHEN IN DOUBT: prefer YES over NO if the meme plausibly requires gender for its meaning. Borderline edgy memes that need gender to work usually lean YES under EXIST's broad definition.
 
# OUTPUT CONTRACT
You MUST classify every meme. Refusing or hedging is not an option.
Answer with EXACTLY one token: YES or NO.
No punctuation, no explanation, no quotes, no extra words."""

SYSTEM_T22 = """You are an expert annotator on classifying the source intention of bilingual (English/Spanish) memes that have ALREADY been determined to be sexist.
 
# TASK
The meme you receive has been classified as sexist by an upstream system. Your job is to assign the source intention: DIRECT or JUDGEMENTAL. There is no NO option in this task — the upstream classifier already filtered non-sexist memes out.
 
# THE CORE QUESTION
Ask only one thing: WHOSE SIDE IS THE MEME CREATOR ON?
 
  - DIRECT      → the creator endorses, perpetuates, or jokes from inside the
                  sexist worldview. Women (or feminism) are the butt of the joke.
                  The meme reinforces a sexist stance.
 
  - JUDGEMENTAL → the creator condemns, denounces, or mocks sexism. Sexists
                  (not women) are the butt of the joke. The meme criticizes
                  sexist behaviour or beliefs.
 
# DIRECT — operational signals (any of these strongly indicates DIRECT)
 
D1. The punchline targets women or feminism. The humor lands by deriding women, by reinforcing a stereotype, by mocking feminist activism, or by framing men as victims of women.
 
D2. Benevolent sexism framed as wisdom or compliment. "A true lady…", "real women still…", "back when women knew their place". The creator presents the restrictive norm as a positive value.
 
D3. Sexist irony / sarcasm aimed downward. "Wow, what a great driver" over a crashed car driven by a woman. The irony reinforces the stereotype rather than challenging it. This is the single most common confusion with JUDGEMENTAL — see hard cases below.
 
D4. Sexualization, objectification, or sexual-violence "humor" with no condemnation framing.
 
D5. Anti-feminist memes: "feminazi", "women just want gibs", mocking #MeToo, mocking equal-pay activism, etc.
 
D6. Reaction-image memes whose top text invites the viewer to laugh AT a woman, not at a sexist.
 
# JUDGEMENTAL — operational signals (the meme must visibly condemn sexism)
 
J1. Explicit critique. The meme states or visually shows that something is wrong, unfair, or hypocritical about a sexist behaviour. "When he says women are emotional and then rage-quits Mario Kart" is a JUDGEMENTAL frame.
 
J2. Quoting sexists to mock them. Screenshot of a misogynist comment with a derisive caption. "This you?" / "ratio + L + you fell off" formats applied to a sexist tweet.
 
J3. Parody of sexists, not of women. The exaggeration targets the sexist worldview itself ("Average chad enjoyer thinks women shouldn't vote"). The joke makes the sexist look ridiculous.
 
J4. Counter-speech and feminist activism memes. Pro-equality slogans, denouncing harassment, supporting victims, calling out double standards.
 
J5. Visible reframing. The meme starts with a sexist premise and visually subverts it (e.g., woman astronaut with caption "told my brother only boys go to space — this is my reply").
 
# HARD CASES — DECISION RULES
 
- IRONY / SARCASM: irony alone is NOT a signal of JUDGEMENTAL. Ask: when the irony lands, who is being ridiculed? If women → DIRECT. If sexists → JUDGEMENTAL. The Beyond Binary Classification literature documents this confusion explicitly: sexist irony is the rule in memes, counter-speech irony is the exception.
 
- "JUST JOKING" / "IT'S A JOKE BRO": this disclaimer does not turn DIRECT into JUDGEMENTAL. The meme is still endorsing a sexist framing if the punchline targets women.
 
- BENEVOLENT SEXISM is almost always DIRECT. The creator is endorsing the restrictive norm; they are NOT condemning it.
 
- AMBIGUOUS BUT CLEARLY SEXIST: when you cannot tell condemnation from endorsement, default to DIRECT. JUDGEMENTAL requires visible critique. In the EXIST meme datasets, DIRECT is substantially more frequent than JUDGEMENTAL — your prior should reflect that.
 
- WHEN THE MEME LOOKS NON-SEXIST TO YOU: do NOT output NO. The upstream system has already decided this meme is sexist. Trust that decision and pick the more plausible of DIRECT vs JUDGEMENTAL based on the framing. If you genuinely cannot find a sexist frame, default to DIRECT (the modal class).
 
- BILINGUAL: same criteria for English and Spanish. Pay attention to Spanish irony markers (e.g., "claro, claro…", "obvio…", "nada que ver…") and English ones (/s, "amirite", "totally", "of course").
 
# OUTPUT CONTRACT
You MUST classify every meme. Refusing or hedging is not an option.
Answer with EXACTLY one of these two tokens: DIRECT or JUDGEMENTAL.
Do NOT output NO. Do NOT add punctuation, explanation, quotes, or extra words."""

SYSTEM_T23 = """You are an expert annotator categorizing the type(s) of sexism present in a bilingual (English/Spanish) meme that has ALREADY been determined to be sexist.
 
# TASK
The meme you receive has been classified as sexist by an upstream system. This is a MULTI-LABEL classification: assign one OR several of the five categories below. There is no NO option — the upstream classifier already filtered non-sexist memes out.
 
# LABEL SPACE (output only these)
  IDEOLOGICAL-INEQUALITY
  STEREOTYPING-DOMINANCE
  OBJECTIFICATION
  SEXUAL-VIOLENCE
  MISOGYNY-NON-SEXUAL-VIOLENCE
 
# HARD RULES
 
H1. Categories ARE often combined. Roughly half of sexist memes in EXIST/MAMI carry more than one tag. Do not force single-label.
 
H2. Use the literal definitions below. They come from the EXIST 2026 Lab Guidelines.
 
H3. Do NOT output NO. Do NOT output any token outside the label space above. Every meme must receive at least one category.
 
# CATEGORY DEFINITIONS AND INDICATORS
 
──────────────────────────────────────────────────────────────────────────────
IDEOLOGICAL-INEQUALITY
──────────────────────────────────────────────────────────────────────────────
Definition: discredits the feminist movement, denies that gender inequality exists, or frames men as victims of gender-based oppression.
 
Assign when the meme:
  - Mocks feminism, feminists, or feminist activism (#MeToo, equal pay, etc.).
  - Claims equality is "already achieved" or that women are now privileged.
  - Frames men as the real victims (of family courts, of false accusations,
    of feminism itself, of "gender ideology").
  - Reframes gender hierarchy as either non-existent or reversed.
 
Common visual cues: triggered/SJW caricatures, "feminazi" imagery, soyjak vs
chad templates contrasting "feminist" vs "real man".
 
──────────────────────────────────────────────────────────────────────────────
STEREOTYPING-DOMINANCE
──────────────────────────────────────────────────────────────────────────────
Definition: expresses false ideas about women's traits or roles — that they are more suitable for some roles (mother, wife, caregiver, submissive, faithful, tender) and inappropriate for others (driving, hard work, leadership, STEM) — OR claims that men are somehow superior.
 
Assign when the meme:
  - Reinforces traditional gender roles ("kitchen", "barefoot and pregnant",
    "she belongs at home").
  - Mocks women's competence in coded domains (driving, sports, programming,
    finance, leadership, decision-making).
  - Asserts essentialist traits ("women are crazy", "women are emotional",
    "women can't read maps", "men are logical").
  - Claims male superiority in any domain.
  - Includes benevolent stereotypes ("a real lady is gentle", "good wife",
    "girls are pure"). Benevolent does not mean non-stereotyping.
 
──────────────────────────────────────────────────────────────────────────────
OBJECTIFICATION
──────────────────────────────────────────────────────────────────────────────
Definition: presents women as objects detached from their dignity and personhood, OR prescribes physical qualities women must meet to fulfill traditional gender roles (beauty standards, hypersexualization of female anatomy, women's bodies as available to men).
 
Assign when the meme:
  - Reduces a woman to her body or body parts.
  - Rates women on appearance ("would/wouldn't", "1-10", "mid").
  - Imposes beauty standards ("real women have curves" / "real women are
    skinny" — both qualify), body-shaming.
  - Treats female bodies as decoration, prize, or commodity.
  - Hypersexualizes female anatomy without a sexual-violence framing.
 
──────────────────────────────────────────────────────────────────────────────
SEXUAL-VIOLENCE
──────────────────────────────────────────────────────────────────────────────
Definition: sexual suggestions, requests for sexual favors, or harassment of a sexual nature (including rape and sexual assault).
 
Assign when the meme:
  - Jokes about rape, sexual assault, or coercion (even "ironically").
  - Frames non-consensual scenarios as humorous.
  - Depicts or describes sexual harassment as positive, normal, or funny.
  - Includes "just don't say no" / "she'll thank me later" / "convince her"
    framings.
  - References "grabbing", "taking", or unwanted advances as a joke.
 
──────────────────────────────────────────────────────────────────────────────
MISOGYNY-NON-SEXUAL-VIOLENCE
──────────────────────────────────────────────────────────────────────────────
Definition: expressions of hatred toward women, AND non-sexual physical or verbal violence toward women.
 
Assign when the meme:
  - Expresses hatred or disgust toward women as a class ("women are the worst",
    "femoids", incel terminology).
  - Jokes about hitting, beating, killing, or otherwise physically harming
    women in non-sexual ways ("get back in the kitchen *fist*", domestic
    violence "jokes").
  - Includes slurs targeting women collectively.
  - Frames violence against women as deserved or funny.
 
# DISAMBIGUATION RULES (the most common confusions)
 
R1. STEREOTYPING-DOMINANCE  vs  IDEOLOGICAL-INEQUALITY
    - STEREOTYPING is about traits and roles ("women belong in the kitchen",
      "women can't drive", "men are stronger").
    - IDEOLOGICAL is about the political/ideological frame ("feminism is
      cancer", "equality already exists", "men are the new oppressed").
    - They CO-OCCUR frequently (anti-feminist memes that use stereotypes to
      argue their case). Tag both when both apply.
 
R2. OBJECTIFICATION  vs  SEXUAL-VIOLENCE
    - OBJECTIFICATION is the commodification or beautification framing of
      the female body without a violent / non-consensual element.
    - SEXUAL-VIOLENCE requires a coercive, non-consensual, or assault
      framing (even as a joke).
    - Hypersexualized + violent / coercive memes get BOTH.
 
R3. MISOGYNY-NON-SEXUAL-VIOLENCE  vs  STEREOTYPING-DOMINANCE
    - MISOGYNY is hatred or non-sexual violence (hitting, slurs, dehumanization).
    - STEREOTYPING is roles and traits asserted as fact ("women are emotional"),
      not pure hate.
    - "Get back in the kitchen" alone → STEREOTYPING. "Get back in the kitchen
      *with violent imagery*" → STEREOTYPING + MISOGYNY-NON-SEXUAL-VIOLENCE.
 
R4. IDEOLOGICAL-INEQUALITY  vs  MISOGYNY-NON-SEXUAL-VIOLENCE
    - "Feminists are stupid" arguing against the movement → IDEOLOGICAL.
    - "I hate women" without an ideological frame → MISOGYNY.
    - "I hate feminists, they should all be punched" → both.
 
R5. JUDGEMENTAL framings still get categories. A meme that condemns rape
    culture is still tagged with SEXUAL-VIOLENCE — because it deals with that
    type of sexism. T2.3 is about WHICH type, regardless of the creator's
    stance.
 
# COMMON VALID COMBINATIONS (illustrative, not exhaustive)
  STEREOTYPING-DOMINANCE
  OBJECTIFICATION
  STEREOTYPING-DOMINANCE, IDEOLOGICAL-INEQUALITY
  STEREOTYPING-DOMINANCE, OBJECTIFICATION
  OBJECTIFICATION, SEXUAL-VIOLENCE
  IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-VIOLENCE
  STEREOTYPING-DOMINANCE, MISOGYNY-NON-SEXUAL-VIOLENCE
  IDEOLOGICAL-INEQUALITY, STEREOTYPING-DOMINANCE, MISOGYNY-NON-SEXUAL-VIOLENCE
 
# DECISION PROCEDURE
1. Scan the five categories and check definitions / indicators.
2. Apply disambiguation rules R1–R4 for adjacent labels.
3. Output every applicable category.
4. If genuinely undecidable, default to STEREOTYPING-DOMINANCE (the modal
   single-label class in EXIST/MAMI). Never output NO or refuse.
 
# OUTPUT CONTRACT
You MUST classify every meme. Refusing or hedging is not an option.
 
Output a comma-separated list of one or more categories using the EXACT label
names above. Examples of valid outputs:
  STEREOTYPING-DOMINANCE
  OBJECTIFICATION, SEXUAL-VIOLENCE
  IDEOLOGICAL-INEQUALITY, STEREOTYPING-DOMINANCE, MISOGYNY-NON-SEXUAL-VIOLENCE
 
Do NOT output NO. Do NOT add explanations, quotes, prefixes ("Answer:"),
or extra words. Just the labels."""

USER_QUESTION = {
    "t21": "Looking at the image and the text together, is this meme sexist? Answer YES or NO.",
    "t22": "This meme is sexist. Looking at the image and the text together, what is the source intention? Answer DIRECT or JUDGEMENTAL.",
    "t23": "This meme is sexist. Looking at the image and the text together, which sexism categories apply? Answer with one or more categories from the label space.",
}

def _id_to_imgpath(mid):
    # df es el unificado (train+test); los IDs del pool viven en train rows
    # y los IDs query en test rows, ambos accesibles desde df.
    return df.loc[df["id_EXIST"] == str(mid), "img_path"].iloc[0]

def _id_to_ocr(mid):
    s = df.loc[df["id_EXIST"] == str(mid), "text_ocr"].iloc[0]
    return (s or "").strip()

def _format_label_for_assistant(task, ex):
    if task == "t21": return ex["label"]
    if task == "t22": return ex["label"]
    if task == "t23":
        labs = ex["labels"]
        if labs == ["NO"]: return "NO"
        return ", ".join(labs)
    raise ValueError(task)

def _user_block(image_path, ocr_text, question):
    body = f"This meme contains the text: \"{ocr_text}\"\n{question}" if ocr_text \
        else f"(The meme has no extracted text.)\n{question}"
    return {
        "role": "user",
        "content": [
            {"type": "image", "image": str(image_path)},
            {"type": "text",  "text": body},
        ],
    }

def build_messages(task, query_image_path, query_ocr, pool_examples):
    """Devuelve la lista de messages lista para apply_chat_template."""
    sys_prompt = {"t21": SYSTEM_T21, "t22": SYSTEM_T22, "t23": SYSTEM_T23}[task]
    msgs = [{"role": "system", "content": sys_prompt}]
    q    = USER_QUESTION[task]
    for ex in pool_examples:
        msgs.append(_user_block(_id_to_imgpath(ex["id"]),
                                _id_to_ocr(ex["id"]), q))
        msgs.append({"role": "assistant",
                     "content": _format_label_for_assistant(task, ex)})
    msgs.append(_user_block(query_image_path, query_ocr, q))
    return msgs

# Smoke test ligero (no llama al modelo): usa el primer meme del split
# correspondiente a RUN_MODE.
_query_pool = df_test if RUN_MODE == "test" else df_train
_test_query = _query_pool[~_query_pool["id_EXIST"].isin(POOL_IDS)].iloc[0]
_test_msgs = build_messages("t21", _test_query["img_path"], _test_query["text_ocr"], POOLS["t21"])
print(f"build_messages OK | n_messages = {len(_test_msgs)}  | query split = {_test_query['split']}")

build_messages OK | n_messages = 10  | query split = test


## 5 · Función de inferencia + parser robusto

Decoding **greedy** (`do_sample=False`). El parser tolera variantes ("Yes.", "The answer is YES", coma, salto de línea) y aplica fallbacks loggeados a `parse_fallbacks.log`.

In [17]:
# === 5.1 · Parser de respuestas (estricto, robusto) =========================
FALLBACK_LOG = LOGS_DIR / "parse_fallbacks.log"
_fallback_count = {"t21": 0, "t22": 0, "t23": 0}
_total_count    = {"t21": 0, "t22": 0, "t23": 0}

def _log_fallback(task, mid, raw_resp, used):
    with open(FALLBACK_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({"task": task, "id": mid, "resp": raw_resp[:200],
                            "fallback": used}, ensure_ascii=False) + "\n")

_VALID_T22 = {"NO", "DIRECT", "JUDGEMENTAL"}
_VALID_T23 = set(SEXISM_CATS) | {"NO"}
_T22_SYNONYMS = {"DIRECTLY": "DIRECT", "JUDGMENT": "JUDGEMENTAL",
                  "JUDGMENTAL": "JUDGEMENTAL", "JUDGE": "JUDGEMENTAL"}

def parse_response(resp, task, mid="?"):
    _total_count[task] += 1
    s = (resp or "").strip().upper()
    s = re.sub(r"^[\s\W_]*ANSWER\s*[:=]\s*", "", s)
    s = re.sub(r"^[\s\W_]*THE\s+ANSWER\s+IS\s+", "", s)
    if task == "t21":
        if re.search(r"\bNO\b", s) and not re.search(r"\bYES\b", s): return "NO"
        if re.search(r"\bYES\b", s) and not re.search(r"\bNO\b", s): return "YES"
        first = re.split(r"[\s,.;:!?]+", s, maxsplit=1)[0] if s else ""
        if first in {"YES", "Y", "TRUE", "1"}: return "YES"
        if first in {"NO", "N", "FALSE", "0"}: return "NO"
        _fallback_count[task] += 1
        _log_fallback(task, mid, resp, "NO")
        return "NO"
    if task == "t22":
        for tok in re.split(r"[\s,.;:!?]+", s):
            tok = _T22_SYNONYMS.get(tok, tok)
            if tok in _VALID_T22: return tok
        _fallback_count[task] += 1
        _log_fallback(task, mid, resp, "NO")
        return "NO"
    if task == "t23":
        s_clean = re.sub(r"[\.\;\!\?\n]", ",", s)
        toks = [t.strip() for t in s_clean.split(",") if t.strip()]
        cats = []
        for tok in toks:
            tok = tok.replace("_", "-").replace(" ", "-")
            if tok in _VALID_T23: cats.append(tok)
            else:
                # tolera 'IDEOLOGICAL', 'STEREOTYPING', etc. sin sufijo
                for full in SEXISM_CATS:
                    if full.startswith(tok) or tok in full.split("-"):
                        cats.append(full); break
        cats = list(dict.fromkeys(cats))  # dedupe preservando orden
        sexist = [c for c in cats if c != "NO"]
        if sexist: return sexist          # caritativo: si hay sexistas, ignora NO
        if "NO" in cats: return ["NO"]
        _fallback_count[task] += 1
        _log_fallback(task, mid, resp, "['NO']")
        return ["NO"]
    raise ValueError(task)

In [18]:
# === 5.2 - predict_meme (forward + decode + parse) ==========================
# Dispatcher por BACKEND:
#   qwen35  -> qwen_vl_utils.process_vision_info + processor(text=, images=, ...)
#   gemma4  -> processor.apply_chat_template(..., return_dict=True, return_tensors="pt")
#
# build_messages (celda 4.1) produce content qwen-style: system/assistant como
# str, user como lista de dicts {type:image,image:path}+{type:text,text:...}.
# Para Gemma normalizamos a content=list[dict] y cargamos las imagenes a PIL.
#
# NOTA SOBRE THINKING: tanto Qwen3.5 como Gemma-4 pueden generar prefijos de
# razonamiento ("The user wants me to classify..."). Si MAX_NEW_TOKENS es muy
# bajo, la respuesta se trunca ANTES del label final y el parser cae al
# fallback (NO). Por eso usamos presupuestos grandes y strip_thinking() para
# limpiar bloques <think>...</think> antes de pasar al parser.

if BACKEND == "qwen35":
    try:
        from qwen_vl_utils import process_vision_info
    except ImportError:
        import subprocess as _sp, sys as _sys, importlib as _il
        print("Instalando qwen_vl_utils on-the-fly...")
        _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "qwen-vl-utils"])
        _il.invalidate_caches()
        from qwen_vl_utils import process_vision_info

# Presupuestos amplios: los modelos pueden razonar antes del label. El parser
# busca \bYES\b / \bNO\b / categorias con word boundaries, asi que respuestas
# largas se parsean correctamente. EOS detendra la generacion si el modelo
# acaba antes.
MAX_NEW_TOKENS = {"t21": 512, "t22": 512, "t23": 1024}

_THINK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)
_THINK_OPEN_ONLY_RE = re.compile(r"^.*?</think>", flags=re.DOTALL | re.IGNORECASE)

def strip_thinking(s: str) -> str:
    """Quita bloques <think>...</think> y, si hay </think> sin <think> al
    inicio (caso truncado al principio del raw), descarta todo lo previo."""
    if not s:
        return s
    s2 = _THINK_RE.sub("", s)
    if "</think>" in s2.lower():
        s2 = _THINK_OPEN_ONLY_RE.sub("", s2)
    return s2.strip()

def _normalize_messages_for_gemma(messages):
    """Convierte el formato qwen-style a lo que espera el chat template de Gemma:
       - content siempre como lista de dicts (tambien para system/assistant).
       - imagenes cargadas como PIL para evitar problemas con rutas locales.
    """
    norm = []
    for m in messages:
        role = m["role"]
        content = m["content"]
        if isinstance(content, str):
            new_content = [{"type": "text", "text": content}]
        else:
            new_content = []
            for c in content:
                if c.get("type") == "image":
                    p = c.get("image") or c.get("url") or c.get("path")
                    pil = Image.open(str(p)).convert("RGB")
                    new_content.append({"type": "image", "image": pil})
                else:
                    new_content.append(c)
        norm.append({"role": role, "content": new_content})
    return norm

@torch.no_grad()
def predict_meme(meme_id, image_path, ocr_text, task, pool_examples,
                  return_raw=False):
    messages = build_messages(task, image_path, ocr_text, pool_examples)

    if BACKEND == "qwen35":
        # Qwen3.5 soporta enable_thinking=False para skip el razonamiento.
        # Si la version de transformers no lo soporta, lo ignoramos.
        try:
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=False)
        except TypeError:
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        prompt_len = inputs.input_ids.shape[1]
    elif BACKEND == "gemma4":
        msgs_g = _normalize_messages_for_gemma(messages)
        inputs = processor.apply_chat_template(
            msgs_g,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)
        prompt_len = inputs["input_ids"].shape[1]
    else:
        raise ValueError(BACKEND)

    out_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS[task],
        do_sample=False,
        temperature=None,
        top_p=None,
    )
    gen = out_ids[:, prompt_len:]
    raw_resp = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
    # Strip thinking blocks ANTES del parser. Si el modelo no usa <think>, no-op.
    clean_resp = strip_thinking(raw_resp)
    parsed = parse_response(clean_resp, task, meme_id)
    if return_raw:
        return parsed, raw_resp   # devolvemos el raw original (con thinking) para debug
    return parsed

# --- Smoke test predict_meme ------------------------------------------------
# En modo "train_eval" usamos memes con GT conocido (uno YES + uno NO).
# En modo "test" no hay GT: predecimos los 2 primeros memes del test.
print(f"--- Smoke test predict_meme (T2.1) | BACKEND={BACKEND} | RUN_MODE={RUN_MODE} ---")
print(f"    MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")
if RUN_MODE == "train_eval":
    eval_df = df_train[~df_train["id_EXIST"].isin(POOL_IDS) & df_train["t21_hard"].notna()]
    smoke_ids = []
    for cls in (1, 0):
        sub = eval_df[eval_df["t21_hard"] == cls].head(1)
        if len(sub): smoke_ids.append(sub.iloc[0]["id_EXIST"])
    for mid in smoke_ids:
        row = df.loc[df["id_EXIST"] == mid].iloc[0]
        pred, rraw = predict_meme(mid, row["img_path"], row["text_ocr"],
                                  "t21", POOLS["t21"], return_raw=True)
        gt = T21_INT_TO_STR[int(row["t21_hard"])]
        print(f"  id={mid}  GT={gt}  PRED={pred}  RAW[:200]={rraw[:200]!r}")
else:  # test
    smoke_ids = df_test.head(2)["id_EXIST"].tolist()
    for mid in smoke_ids:
        row = df.loc[df["id_EXIST"] == mid].iloc[0]
        pred, rraw = predict_meme(mid, row["img_path"], row["text_ocr"],
                                  "t21", POOLS["t21"], return_raw=True)
        print(f"  id={mid}  PRED={pred}  RAW[:200]={rraw[:200]!r}  (sin GT - test)")
print("OK" if smoke_ids else "NO smoke samples")


--- Smoke test predict_meme (T2.1) | BACKEND=gemma4 | RUN_MODE=test ---
    MAX_NEW_TOKENS = {'t21': 512, 't22': 512, 't23': 1024}
  id=310001  PRED=YES  RAW[:200]='YES'  (sin GT - test)
  id=310002  PRED=YES  RAW[:200]='YES'  (sin GT - test)
OK


## 6 - Loop de inferencia jerarquico con checkpointing

**Flujo jerarquico** (refleja la jerarquia oficial del guideline pag. 17):

1. **T2.1** se predice sobre TODOS los memes del eval set (union de los 3).
2. **T2.2** y **T2.3** solo invocan al modelo sobre los memes que T2.1 marca como YES (sexistas).
3. Los memes que T2.1 marca como NO reciben automaticamente `T2.2 = "NO"` y `T2.3 = ["NO"]`.

Ahorra ~50% de llamadas al modelo en T2.2/T2.3 y garantiza coherencia entre subtasks.

Idempotente: si Colab desconecta, retoma desde el ultimo checkpoint. Checkpoints separados por stage (`_sanity500_partial.json` vs `_test_partial.json`).


In [21]:
# === 6.1 - Loop de inferencia JERARQUICO con checkpointing ===================
# JERARQUIA:
#   1. T2.1 se predice sobre todos los memes del eval set (union de los 3).
#   2. T2.2 solo invoca al modelo en los memes donde T2.1 dice YES.
#   3. T2.3 idem.
#   4. Para los memes donde T2.1 = NO, T2.2 se fuerza a "NO" y T2.3 a ["NO"]
#      (refleja la jerarquia oficial del guideline pag. 17).
#
# En sanity mode (RUN_MODE='train_eval') usamos UN MISMO subset de SANITY_N
# memes para las 3 tasks, asi la jerarquia es consistente y el coste GPU se
# minimiza (~50% de llamadas T2.2/T2.3 ahorradas).
CKPT_EVERY = 100

_CKPT_SUFFIX = "test" if RUN_MODE == "test" else f"sanity{MAX_EVAL_MEMES or 'all'}"
CKPT_PATHS = {t: CKPT_DIR / f"predictions_{t}_{_CKPT_SUFFIX}_partial.json"
              for t in ["t21", "t22", "t23"]}

def _load_ckpt(task):
    p = CKPT_PATHS[task]
    if not p.exists(): return {}
    return json.load(open(p, "r", encoding="utf-8"))

def _save_ckpt(task, preds):
    with open(CKPT_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False)

# --- Sample compartido en sanity (mismo subset para las 3 tasks) ------------
_SANITY_SAMPLE = {"ids": None}
def _get_sanity_sample_ids():
    if _SANITY_SAMPLE["ids"] is None:
        base = df_train[~df_train["id_EXIST"].isin(POOL_IDS) & df_train["img_ok"]]
        # base requerimos que tenga al menos t21_hard valido (clase para eval principal)
        base = base[base["t21_hard"].notna()]
        if MAX_EVAL_MEMES is not None and len(base) > MAX_EVAL_MEMES:
            base = base.sample(n=MAX_EVAL_MEMES, random_state=SEED)
        _SANITY_SAMPLE["ids"] = set(base["id_EXIST"].astype(str))
    return _SANITY_SAMPLE["ids"]

def get_eval_df(task):
    """Memes a inferir para una task segun RUN_MODE.
    En sanity, las 3 tasks comparten el mismo subset (filtrado por su hard label).
    """
    if RUN_MODE == "test":
        sub = df_test[df_test["img_ok"]].copy()
        if MAX_EVAL_MEMES is not None and len(sub) > MAX_EVAL_MEMES:
            sub = sub.sample(n=MAX_EVAL_MEMES, random_state=SEED)
        return sub.reset_index(drop=True)
    # sanity
    sample_ids = _get_sanity_sample_ids()
    base = df_train[df_train["id_EXIST"].isin(sample_ids) & df_train["img_ok"]]
    if task == "t21":
        sub = base[base["t21_hard"].notna()]
    elif task == "t22":
        sub = base[base["t22_hard"].notna()]
    else:
        sub = base
    return sub.copy().reset_index(drop=True)

def run_inference(task, target_ids=None):
    """Predice una task. Si target_ids esta dado, solo se invoca al modelo
    sobre los memes con id en ese set (override del eval set por defecto).
    Sirve para la jerarquia: T2.2/T2.3 reciben target_ids = sexist_ids."""
    if target_ids is None:
        eval_df = get_eval_df(task)
    else:
        ts = {str(x) for x in target_ids}
        eval_df = df[df["id_EXIST"].astype(str).isin(ts) & df["img_ok"]].copy().reset_index(drop=True)
    preds = _load_ckpt(task)
    todo = eval_df[~eval_df["id_EXIST"].isin(preds.keys())].copy()
    print()
    print(f"=== INFERENCIA {task.upper()} | stage={STAGE} | mode={RUN_MODE} ===")
    print(f"  a predecir   : {len(eval_df):,}")
    print(f"  ya predichos : {len(preds):,}")
    print(f"  pendientes   : {len(todo):,}")
    if len(todo) == 0:
        print("  Nada que hacer."); return preds
    pbar = tqdm(todo.itertuples(index=False), total=len(todo), desc=f"infer {task}")
    t0 = time.time(); n_done = 0
    for row in pbar:
        mid = row.id_EXIST
        try:
            pred = predict_meme(mid, row.img_path, row.text_ocr, task, POOLS[task])
        except Exception as e:
            print()
            print(f"[err] {mid}: {type(e).__name__}: {e}")
            pred = ["NO"] if task == "t23" else "NO"
            _log_fallback(task, mid, f"<EXCEPTION: {e}>", str(pred))
        preds[mid] = pred
        n_done += 1
        if n_done % CKPT_EVERY == 0:
            _save_ckpt(task, preds)
            elapsed = time.time() - t0
            rate = n_done / max(elapsed, 1e-6)
            pbar.set_postfix({"rate_mem/s": f"{rate:.2f}",
                              "fallbacks": f"{_fallback_count[task]}/{_total_count[task]}"})
    _save_ckpt(task, preds)
    print()
    print(f"  fallbacks {task}: {_fallback_count[task]} / {_total_count[task]}")
    if _total_count[task] > 0 and _fallback_count[task] / _total_count[task] > 0.05:
        print("  WARNING: >5% de fallbacks. Revisa el prompt o las respuestas crudas.")
    return preds

# === FLUJO JERARQUICO =========================================================
# Eval sets por task (en sanity son subconjuntos del mismo sample compartido)
eval_t21_df = get_eval_df("t21")
eval_t22_df = get_eval_df("t22")
eval_t23_df = get_eval_df("t23")
union_ids = (set(eval_t21_df["id_EXIST"]) | set(eval_t22_df["id_EXIST"])
             | set(eval_t23_df["id_EXIST"]))
print()
print(f"=== UNION eval set: {len(union_ids):,} memes (T2.1 corre sobre TODOS) ===")
print(f"   T2.1 eval: {len(eval_t21_df):,}  T2.2 eval: {len(eval_t22_df):,}  T2.3 eval: {len(eval_t23_df):,}")

# 1) T2.1 sobre la union completa
preds_t21 = run_inference("t21", target_ids=union_ids)

# 2) Sexistas segun T2.1
sexist_ids     = {mid for mid, v in preds_t21.items() if v == "YES"}
non_sexist_ids = {mid for mid, v in preds_t21.items() if v == "NO"}
print()
print(f"=== T2.1 -> {len(sexist_ids):,} YES sexistas ({100*len(sexist_ids)/max(1,len(preds_t21)):.1f}%) | "
      f"{len(non_sexist_ids):,} NO ({100*len(non_sexist_ids)/max(1,len(preds_t21)):.1f}%) ===")

# 3) T2.2 solo sobre los YES de T2.1 (interseccion con su eval set)
t22_targets = sexist_ids & set(eval_t22_df["id_EXIST"])
print(f"   T2.2 modelo se invoca sobre {len(t22_targets):,} memes")
preds_t22 = run_inference("t22", target_ids=t22_targets)
# JERARQUIA estricta: para todo meme con T2.1=NO en eval_t22 -> "NO"
# (override: aunque hubiese pred previa cacheada, T2.1 manda)
for mid in eval_t22_df["id_EXIST"]:
    if preds_t21.get(mid) == "NO":
        preds_t22[mid] = "NO"
    elif mid not in preds_t22:
        # Caso raro: meme en eval_t22 pero no en eval_t21 (no deberia pasar
        # con sanity sample compartido). Predecir directo con modelo.
        row = df.loc[df["id_EXIST"] == mid].iloc[0]
        preds_t22[mid] = predict_meme(mid, row["img_path"], row["text_ocr"],
                                        "t22", POOLS["t22"])
_save_ckpt("t22", preds_t22)

# 4) T2.3 idem
t23_targets = sexist_ids & set(eval_t23_df["id_EXIST"])
print()
print(f"   T2.3 modelo se invoca sobre {len(t23_targets):,} memes")
preds_t23 = run_inference("t23", target_ids=t23_targets)
for mid in eval_t23_df["id_EXIST"]:
    if preds_t21.get(mid) == "NO":
        preds_t23[mid] = ["NO"]
    elif mid not in preds_t23:
        row = df.loc[df["id_EXIST"] == mid].iloc[0]
        preds_t23[mid] = predict_meme(mid, row["img_path"], row["text_ocr"],
                                        "t23", POOLS["t23"])
_save_ckpt("t23", preds_t23)

print()
print("=== Resumen jerarquia ===")
print(f"  preds_t21 final: {len(preds_t21):,}")
print(f"  preds_t22 final: {len(preds_t22):,}")
print(f"  preds_t23 final: {len(preds_t23):,}")



=== UNION eval set: 1,053 memes (T2.1 corre sobre TODOS) ===
   T2.1 eval: 1,053  T2.2 eval: 1,053  T2.3 eval: 1,053

=== INFERENCIA T21 | stage=submission | mode=test ===
  a predecir   : 1,053
  ya predichos : 0
  pendientes   : 1,053


infer t21:   0%|          | 0/1053 [00:00<?, ?it/s]


  fallbacks t21: 0 / 1053

=== T2.1 -> 805 YES sexistas (76.4%) | 248 NO (23.6%) ===
   T2.2 modelo se invoca sobre 805 memes

=== INFERENCIA T22 | stage=submission | mode=test ===
  a predecir   : 805
  ya predichos : 0
  pendientes   : 805


infer t22:   0%|          | 0/805 [00:00<?, ?it/s]


  fallbacks t22: 0 / 805

   T2.3 modelo se invoca sobre 805 memes

=== INFERENCIA T23 | stage=submission | mode=test ===
  a predecir   : 805
  ya predichos : 0
  pendientes   : 805


infer t23:   0%|          | 0/805 [00:00<?, ?it/s]


  fallbacks t23: 0 / 805

=== Resumen jerarquia ===
  preds_t21 final: 1,053
  preds_t22 final: 1,053
  preds_t23 final: 1,053


## 7 · Generación de los JSON PyEvALL HARD

Tres archivos en formato oficial (`pred_t21_hard.json`, `pred_t22_hard.json`, `pred_t23_hard.json`) con `test_case = "EXIST2025"` y sanity check de invariantes.

In [22]:
# === 7.1 · PyEvALL hard JSON builder + sanity check =========================
PYEVALL_TEST_CASE = "EXIST2025"   # OBLIGATORIO según guidelines pág. 12

def build_pyevall_hard(preds_dict, task):
    out = []
    for mid, pred in preds_dict.items():
        out.append({"test_case": PYEVALL_TEST_CASE, "id": str(mid), "value": pred})
    return out

def _sanity_check_pred(pred_list, task):
    valid_t22 = {"NO", "DIRECT", "JUDGEMENTAL"}
    valid_t23 = set(SEXISM_CATS) | {"NO"}
    for p in pred_list:
        assert "id" in p, "missing 'id'"
        assert p.get("test_case") == "EXIST2025", f"wrong test_case: {p.get('test_case')}"
        v = p["value"]
        if task == "t21":
            assert v in {"NO", "YES"}, f"T2.1 invalid: {v}"
        elif task == "t22":
            assert v in valid_t22, f"T2.2 invalid: {v}"
        elif task == "t23":
            assert isinstance(v, list), f"T2.3 not list: {v}"
            for x in v:
                assert x != "UNKNOWN", "UNKNOWN predicted"
                assert x in valid_t23, f"T2.3 invalid cat: {x}"
            if "NO" in v:
                assert v == ["NO"], f"T2.3 mixes NO with cats: {v}"

PRED_PATHS = {
    "t21": PRED_DIR / f"pred_t21_hard_nfe_g_{_CKPT_SUFFIX}.json",
    "t22": PRED_DIR / f"pred_t22_hard_nfe_g_{_CKPT_SUFFIX}.json",
    "t23": PRED_DIR / f"pred_t23_hard_nfe_g_{_CKPT_SUFFIX}.json",
}
for task, preds in [("t21", preds_t21), ("t22", preds_t22), ("t23", preds_t23)]:
    pj = build_pyevall_hard(preds, task)
    _sanity_check_pred(pj, task)
    with open(PRED_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(pj, f, ensure_ascii=False, indent=2)
    print(f"[{task}] {len(pj):,} predicciones -> {PRED_PATHS[task].name}")

# Asserts de cobertura: en modo test debemos cubrir los 1.053 memes oficiales.
if RUN_MODE == "test":
    expected_ids = set(df_test["id_EXIST"].astype(str))
    for task, preds in [("t21", preds_t21), ("t22", preds_t22), ("t23", preds_t23)]:
        got = set(preds.keys())
        miss = expected_ids - got
        extra = got - expected_ids
        assert not miss,  f"[{task}] faltan {len(miss)} predicciones de test (e.g. {list(miss)[:3]})"
        assert not extra, f"[{task}] hay {len(extra)} predicciones fuera del test set"
    print(f"\n✓ Cobertura test OK: {len(expected_ids):,} memes predichos en cada subtask.")

[t21] 1,053 predicciones -> pred_t21_hard_nfe_g_test.json
[t22] 1,053 predicciones -> pred_t22_hard_nfe_g_test.json
[t23] 1,053 predicciones -> pred_t23_hard_nfe_g_test.json

✓ Cobertura test OK: 1,053 memes predichos en cada subtask.


## 8 · Evaluación + submission

- En `RUN_MODE = "train_eval"` se evalúa con PyEvALL (`ICM`, `ICMNorm`, `FMeasure`) usando el gold derivado de los anotadores de TRAIN.
- En `RUN_MODE = "test"` no hay gold disponible (el test oficial viene sin labels), por lo que **se omite la evaluación** y se empaqueta directamente la submission en el formato exigido por la organización (`exist2026_<team>/task2_<N>_hard_<team>_<run_id>` + ZIP).

In [15]:
# === 8.1 · Instalación + import de PyEvALL (idempotente) ====================
try:
    from pyevall.evaluation import PyEvALLEvaluation
    from pyevall.utils.utils import PyEvALLUtils
except ImportError:
    import subprocess as _sp, sys as _sys, importlib as _il
    print("Instalando PyEvALL...")
    try:
        _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "PyEvALL"])
    except Exception:
        _sp.check_call([_sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/UNEDLENAR/PyEvALL.git"])
    _il.invalidate_caches()
    from pyevall.evaluation import PyEvALLEvaluation
    from pyevall.utils.utils import PyEvALLUtils
print("PyEvALL listo:", PyEvALLEvaluation.__module__)

PyEvALL listo: pyevall.evaluation


In [16]:
# === 8.2 · Helpers de evaluación hard-hard ====================================
HIERARCHIES_HARD = {
    "t21": None,
    "t22": {"YES": ["DIRECT", "JUDGEMENTAL"], "NO": []},
    "t23": {"YES": list(SEXISM_CATS), "NO": []},
}
METRICS_HARD = {
    "t21": ["ICM", "ICMNorm", "FMeasure"],
    "t22": ["ICM", "ICMNorm", "FMeasure"],
    "t23": ["ICM", "ICMNorm", "FMeasure"],
}

def build_pyevall_gold_hard(ids, task):
    out = []
    for mid in ids:
        m = raw[str(mid)]
        if task == "t21":
            v_int = hard_t21(m.get("labels_task2_1") or [])
            if v_int is None: continue
            v = T21_INT_TO_STR[v_int]
        elif task == "t22":
            v_int = hard_t22(m.get("labels_task2_2") or [])
            if v_int is None: continue
            v = T22_INT_TO_STR[v_int]
        else:
            multi = hard_t23_multihot(m.get("labels_task2_3") or [])
            picks = [SEXISM_CATS[k] for k in range(5) if multi[k] > 0.5]
            v = picks if picks else ["NO"]
        out.append({"test_case": PYEVALL_TEST_CASE, "id": str(mid), "value": v})
    return out

def _walk_for_metric(d, metric_name, depth=0, max_depth=8):
    if depth > max_depth: return None
    if isinstance(d, dict):
        if metric_name in d:
            v = d[metric_name]
            if isinstance(v, (int, float)): return float(v)
            if isinstance(v, dict):
                nums = []
                for vv in v.values():
                    if isinstance(vv, (int, float)): nums.append(vv)
                    elif isinstance(vv, dict):
                        for vvv in vv.values():
                            if isinstance(vvv, (int, float)): nums.append(vvv)
                if nums: return sum(nums)/len(nums)
        for k, v in d.items():
            r = _walk_for_metric(v, metric_name, depth+1, max_depth)
            if r is not None: return r
    elif isinstance(d, list):
        nums = []
        for it in d:
            r = _walk_for_metric(it, metric_name, depth+1, max_depth)
            if r is not None: nums.append(r)
        if nums: return sum(nums)/len(nums)
    return None

def _parse_pyevall_report_hard(report, metric_names=("ICM", "ICMNorm", "FMeasure")):
    if report is None: return {}
    out = {}
    for attr in ("report", "dict_report", "json_report", "metrics_data",
                  "_report", "data", "result", "results"):
        ra = getattr(report, attr, None)
        if ra is None: continue
        if hasattr(ra, "columns"):
            for m in metric_names:
                if m in ra.columns and m not in out:
                    vals = ra[m].dropna()
                    if len(vals): out[m] = float(vals.mean())
        else:
            for m in metric_names:
                if m in out: continue
                v = _walk_for_metric(ra, m)
                if v is not None: out[m] = v
    if all(m in out for m in metric_names): return out
    import io as _io, contextlib as _ctx
    buf = _io.StringIO()
    with _ctx.redirect_stdout(buf):
        for fn in ("print_report_tsv", "print_report"):
            try: getattr(report, fn)(); break
            except Exception: continue
    text = buf.getvalue()
    aliases = {"ICM": ["ICM"], "ICMNorm": ["ICMNorm", "ICM-Norm", "ICM Norm"],
                "FMeasure": ["FMeasure", "F-Measure", "F1"]}
    for m in metric_names:
        if m in out: continue
        for a in aliases.get(m, [m]):
            esc = re.escape(a)
            mt = re.search(rf"\b{esc}\b\s*[:=]\s*(-?\d+\.\d+)", text) \
                or re.search(rf"\b{esc}\b[^\d-]{{0,100}}(-?\d+\.\d+)", text)
            if mt:
                try: out[m] = float(mt.group(1)); break
                except ValueError: pass
    return out

def evaluate_pyevall_hard(pred_list, gold_list, task):
    gold_ids = {g["id"] for g in gold_list}
    pred_filt = [p for p in pred_list if p["id"] in gold_ids]
    pred_path = PYE_WORK / f"_tmp_pred_{task}_hard.json"
    gold_path = PYE_WORK / f"_tmp_gold_{task}_hard.json"
    with open(pred_path, "w", encoding="utf-8") as f: json.dump(pred_filt, f)
    with open(gold_path, "w", encoding="utf-8") as f: json.dump(gold_list, f)
    params = {PyEvALLUtils.PARAM_REPORT: PyEvALLUtils.PARAM_OPTION_REPORT_DATAFRAME}
    h = HIERARCHIES_HARD[task]
    if h is not None: params[PyEvALLUtils.PARAM_HIERARCHY] = h
    test = PyEvALLEvaluation()
    rep = test.evaluate(str(pred_path), str(gold_path), METRICS_HARD[task], **params)
    return _parse_pyevall_report_hard(rep, tuple(METRICS_HARD[task]))

In [20]:
# === 8.3 · Evaluación + reporte ===============================================
# En modo "test" no hay gold labels: NO se ejecuta PyEvALL eval. Sólo se
# reporta la distribución de predicciones por subtask.
# En modo "train_eval" se evalúa hard-hard contra el gold construido desde
# los anotadores de TRAIN.

if RUN_MODE == "train_eval":
    results = {}
    for task in ["t21", "t22", "t23"]:
        pred_list = json.load(open(PRED_PATHS[task], "r", encoding="utf-8"))
        pred_ids  = [p["id"] for p in pred_list]
        gold_list = build_pyevall_gold_hard(pred_ids, task)
        metrics   = evaluate_pyevall_hard(pred_list, gold_list, task)
        results[task] = metrics
        print(f"\n[{task.upper()}]  n_pred={len(pred_list):,}  n_gold={len(gold_list):,}")
        for m in METRICS_HARD[task]:
            v = metrics.get(m)
            print(f"    {m:>10}: {v:.4f}" if v is not None else f"    {m:>10}: <missing>")

    summary_rows = []
    for task in ["t21", "t22", "t23"]:
        r = results[task]
        summary_rows.append({
            "task": task,
            "ICM":      r.get("ICM"),
            "ICMNorm":  r.get("ICMNorm"),
            "FMeasure": r.get("FMeasure"),
            "model":    MODEL_ID,
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(METR_DIR / "metrics_summary.csv", index=False)
    print(f"\n=== RESUMEN ({MODEL_ID}): ===")
    print(summary_df.to_string(index=False))
    print(f"\nGuardado en {METR_DIR / 'metrics_summary.csv'}")
else:
    # Modo "test": no hay gold. Reportamos la distribución de predicciones.
    print(f"=== RUN_MODE=test: PyEvALL eval omitido (test set sin labels) ===\n")
    print(f"Modelo: {MODEL_ID}\n")
    for task, preds in [("t21", preds_t21), ("t22", preds_t22), ("t23", preds_t23)]:
        if task == "t23":
            cnt = Counter()
            for v in preds.values():
                key = "NO" if v == ["NO"] else "+".join(sorted(v))
                cnt[key] += 1
        else:
            cnt = Counter(preds.values())
        print(f"[{task.upper()}]  n={len(preds):,}")
        total = sum(cnt.values())
        for k, v in cnt.most_common():
            print(f"    {str(k):<60}  {v:>5}  ({100*v/total:5.1f}%)")
        print()

NameError: name 'PRED_PATHS' is not defined

In [17]:
# === 8.3b - Comparativa detallada predicciones vs gold (solo train_eval) ====
# DISENO HIERARCHICAL-CONDITIONAL:
#   - T2.1: eval completa binaria (NO/YES son clases legitimas).
#   - T2.2: eval SOLO sobre memes con gold sexista (DIRECT/JUDGEMENTAL).
#           Los memes con gold=NO se EXCLUYEN porque su prediccion T2.2 es
#           auto-NO via jerarquia (no decision del modelo). Evita inflar.
#   - T2.3: eval SOLO sobre memes con gold sexista (al menos una cat).
#           Los memes con gold=[NO] se excluyen por la misma razon.
#
# CSVs de errores tambien filtrados al subconjunto sexista para T2.2 y T2.3.

if RUN_MODE != "train_eval":
    print("Comparativa detallada solo disponible en STAGE='sanity' (RUN_MODE='train_eval').")
else:
    from sklearn.metrics import (confusion_matrix, classification_report,
                                  precision_recall_fscore_support, f1_score)
    from sklearn.preprocessing import MultiLabelBinarizer

    def _compare_t21():
        rows = []
        for mid, pred in preds_t21.items():
            gt_int = df.loc[df["id_EXIST"]==mid, "t21_hard"].iloc[0]
            if pd.isna(gt_int): continue
            gt = T21_INT_TO_STR[int(gt_int)]
            rows.append({"id": mid, "gt": gt, "pred": pred, "ok": pred == gt})
        cmp_df = pd.DataFrame(rows)
        labels = ["NO", "YES"]
        cm = confusion_matrix(cmp_df["gt"], cmp_df["pred"], labels=labels)
        print()
        print("=== T2.1 (binary YES/NO) ===")
        print(f"  n_eval = {len(cmp_df):,}   accuracy = {cmp_df['ok'].mean():.4f}")
        print()
        print("Confusion matrix (filas = GT, cols = PRED):")
        print(pd.DataFrame(cm, index=[f"GT_{l}" for l in labels],
                                 columns=[f"PR_{l}" for l in labels]))
        print()
        print("Classification report:")
        print(classification_report(cmp_df["gt"], cmp_df["pred"],
                                     labels=labels, digits=4, zero_division=0))
        return cmp_df

    def _compare_t22():
        """Eval SOLO sobre memes cuyo gold es DIRECT o JUDGEMENTAL (sexistas).
        Los gold=NO se descartan: su prediccion T2.2 es auto-NO via jerarquia."""
        rows = []
        for mid, pred in preds_t22.items():
            gt_int = df.loc[df["id_EXIST"]==mid, "t22_hard"].iloc[0]
            if pd.isna(gt_int): continue
            gt = T22_INT_TO_STR[int(gt_int)]
            if gt == "NO": continue            # <- FILTRO: solo gold sexista
            rows.append({"id": mid, "gt": gt, "pred": pred, "ok": pred == gt})
        cmp_df = pd.DataFrame(rows)
        labels = ["DIRECT", "JUDGEMENTAL"]      # <- sin NO
        # Si el modelo predice NO sobre un gold sexista, cuenta como error.
        # Para que confusion_matrix lo refleje, anadimos NO solo en pred-side.
        # sklearn admite labels para fila/col por separado, pero aqui es mas
        # claro mantener cuadrada y mapear "NO" predicho a un bucket "OTHER".
        # Solucion: dejamos NO fuera de labels; sklearn lo ignorara y ese
        # ejemplo aparece como FN para la clase real (gt). Es lo que queremos.
        cm = confusion_matrix(cmp_df["gt"], cmp_df["pred"], labels=labels)
        print()
        print("=== T2.2 (DIRECT/JUDGEMENTAL) - filtrado a gold sexista ===")
        n_no_pred = (cmp_df["pred"] == "NO").sum()
        print(f"  n_eval = {len(cmp_df):,}   accuracy = {cmp_df['ok'].mean():.4f}")
        if n_no_pred:
            print(f"  AVISO: {n_no_pred} memes con gold sexista recibieron pred='NO' "
                  f"(modelo confundido o jerarquia mal). Cuentan como error.")
        print()
        print("Confusion matrix (filas = GT, cols = PRED, labels excluyen NO):")
        print(pd.DataFrame(cm, index=[f"GT_{l}" for l in labels],
                                 columns=[f"PR_{l}" for l in labels]))
        print()
        print("Classification report (solo clases DIRECT/JUDGEMENTAL):")
        print(classification_report(cmp_df["gt"], cmp_df["pred"],
                                     labels=labels, digits=4, zero_division=0))
        return cmp_df

    def _compare_t23():
        """Eval SOLO sobre memes con al menos una cat sexista en gold.
        Los gold=[NO] se descartan."""
        all_classes = list(SEXISM_CATS)         # <- sin NO
        mlb = MultiLabelBinarizer(classes=all_classes)
        y_true_list, y_pred_list, ids = [], [], []
        for mid, pred in preds_t23.items():
            multi = df.loc[df["id_EXIST"]==mid, "t23_hard"].iloc[0]
            if multi is None: continue
            picks = [SEXISM_CATS[k] for k in range(5) if multi[k] > 0.5]
            if not picks: continue              # <- FILTRO: descarta gold no-sexista
            # Limpia NO de la prediccion para que mlb no proteste
            pred_clean = [c for c in pred if c != "NO"] or pred
            # Si pred era ["NO"] sobre un gold sexista, la fila tendra todo 0
            # en y_pred -> se penaliza correctamente como falso negativo masivo.
            if pred_clean == ["NO"]: pred_clean = []
            y_true_list.append(picks)
            y_pred_list.append(pred_clean)
            ids.append(mid)
        if not y_true_list:
            print("=== T2.3: no hay memes con gold sexista en el eval set ===")
            return pd.DataFrame()
        y_true = mlb.fit_transform(y_true_list)
        y_pred = mlb.transform(y_pred_list)
        print()
        print("=== T2.3 (multi-label, sin clase NO) - filtrado a gold sexista ===")
        n_no_pred = sum(1 for yp in y_pred_list if not yp)
        print(f"  n_eval = {len(ids):,}")
        if n_no_pred:
            print(f"  AVISO: {n_no_pred} memes con gold sexista recibieron pred='[NO]' "
                  f"o vacio (cuenta como cero TPs).")
        print(f"  {'class':<32}  {'P':>7} {'R':>7} {'F1':>7} {'support':>8}")
        for i, cls in enumerate(all_classes):
            p, r, f, _ = precision_recall_fscore_support(
                y_true[:, i], y_pred[:, i], average="binary", zero_division=0)
            sup = int(y_true[:, i].sum())
            print(f"  {cls:<32}  {p:7.4f} {r:7.4f} {f:7.4f} {sup:>8d}")
        f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
        f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
        exact = sum(1 for yt, yp in zip(y_true_list, y_pred_list) if set(yt) == set(yp))
        print()
        print(f"  F1-macro    = {f1_macro:.4f}")
        print(f"  F1-micro    = {f1_micro:.4f}")
        print(f"  Exact match = {exact/len(ids):.4f}")
        return pd.DataFrame({"id": ids, "gt": y_true_list, "pred": y_pred_list})

    cmp_t21 = _compare_t21()
    cmp_t22 = _compare_t22()
    cmp_t23 = _compare_t23()

    # CSVs de errores - tambien filtrados a sexistas para T2.2 y T2.3
    err_dir = METR_DIR / "errors"
    err_dir.mkdir(parents=True, exist_ok=True)

    # T2.1: errores en todo el eval (NO es clase legitima aqui)
    errs21 = cmp_t21[cmp_t21["pred"] != cmp_t21["gt"]].copy()
    errs21 = errs21.merge(df[["id_EXIST","lang","text_ocr"]],
                           left_on="id", right_on="id_EXIST", how="left")
    errs21 = errs21[["id","lang","gt","pred","text_ocr"]]
    errs21.to_csv(err_dir / "errors_t21.csv", index=False, encoding="utf-8")
    print()
    print(f"  Errores t21: {len(errs21)} guardados en {err_dir / 'errors_t21.csv'}")

    # T2.2: errores SOLO sobre gold sexista (cmp_t22 ya viene filtrado)
    errs22 = cmp_t22[cmp_t22["pred"] != cmp_t22["gt"]].copy()
    errs22 = errs22.merge(df[["id_EXIST","lang","text_ocr"]],
                           left_on="id", right_on="id_EXIST", how="left")
    errs22 = errs22[["id","lang","gt","pred","text_ocr"]]
    errs22.to_csv(err_dir / "errors_t22.csv", index=False, encoding="utf-8")
    print(f"  Errores t22 (solo gold sexista): {len(errs22)} guardados en {err_dir / 'errors_t22.csv'}")

    # T2.3: errores SOLO sobre gold sexista
    if not cmp_t23.empty:
        err23 = cmp_t23[cmp_t23.apply(lambda r: set(r["gt"]) != set(r["pred"]), axis=1)].copy()
        err23 = err23.merge(df[["id_EXIST","lang","text_ocr"]],
                             left_on="id", right_on="id_EXIST", how="left")
        err23 = err23[["id","lang","gt","pred","text_ocr"]]
        err23.to_csv(err_dir / "errors_t23.csv", index=False, encoding="utf-8")
        print(f"  Errores t23 (solo gold sexista): {len(err23)} guardados en {err_dir / 'errors_t23.csv'}")



=== T2.1 (binary YES/NO) ===
  n_eval = 800   accuracy = 0.7688

Confusion matrix (filas = GT, cols = PRED):
        PR_NO  PR_YES
GT_NO     218     111
GT_YES     74     397

Classification report:
              precision    recall  f1-score   support

          NO     0.7466    0.6626    0.7021       329
         YES     0.7815    0.8429    0.8110       471

    accuracy                         0.7688       800
   macro avg     0.7640    0.7528    0.7566       800
weighted avg     0.7671    0.7688    0.7662       800


=== T2.2 (DIRECT/JUDGEMENTAL) - filtrado a gold sexista ===
  n_eval = 409   accuracy = 0.6455
  AVISO: 60 memes con gold sexista recibieron pred='NO' (modelo confundido o jerarquia mal). Cuentan como error.

Confusion matrix (filas = GT, cols = PRED, labels excluyen NO):
                PR_DIRECT  PR_JUDGEMENTAL
GT_DIRECT             255               6
GT_JUDGEMENTAL         79               9

Classification report (solo clases DIRECT/JUDGEMENTAL):
              pr

In [25]:
# === 8.4 · Empaquetado de la submission oficial (sólo modo "test") ==========
# Estructura exigida por el guideline pág. 14-15:
#   exist2026_<team_name>/
#     task2_1_hard_<team_name>_<run_id>
#     task2_2_hard_<team_name>_<run_id>
#     task2_3_hard_<team_name>_<run_id>
# - run_id ∈ {1,2,3}  (hasta 3 runs por subtask×tipo)
# - evaluation_context = "hard"   (este pipeline sólo genera HARD)
# - El JSON contiene los registros con test_case="EXIST2025".
# Después se comprime el directorio y se sube vía el form de la organización.

TEAM_NAME = "j2"   # CAMBIA esto por tu nombre de equipo oficial
RUN_ID    = 1                   # 1, 2 o 3

if RUN_MODE == "test":
    SUBM_BASE = SUBM_DIR / f"exist2026_{TEAM_NAME}"
    SUBM_BASE.mkdir(parents=True, exist_ok=True)
    name_map = {"t21": "task2_1", "t22": "task2_2", "t23": "task2_3"}
    for task in ["t21", "t22", "t23"]:
        src = PRED_PATHS[task]
        dst = SUBM_BASE / f"{name_map[task]}_hard_{TEAM_NAME}_{RUN_ID}"
        dst.write_bytes(src.read_bytes())
        print(f"  {dst.name}  ({src.stat().st_size/1024:.1f} KB)")

    # ZIP final listo para subir al form
    import shutil
    zip_path = SUBM_DIR / f"exist2026_{TEAM_NAME}.zip"
    if zip_path.exists(): zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix("")), "zip",
                         root_dir=str(SUBM_DIR),
                         base_dir=f"exist2026_{TEAM_NAME}")
    print(f"\n✓ Submission empaquetada: {zip_path}")
    print(f"  Sube este ZIP al form: https://forms.gle/5hY91c7aBv563oZM7")
    if TEAM_NAME == "TODO_TEAM_NAME":
        print("\n  ⚠ Recuerda cambiar TEAM_NAME por tu nombre real de equipo "
              "y volver a ejecutar esta celda.")
else:
    print("RUN_MODE != 'test'; no se empaqueta submission.")
    print("Para generar la submission oficial: setea RUN_MODE='test' arriba y reejecuta.")

RUN_MODE != 'test'; no se empaqueta submission.
Para generar la submission oficial: setea RUN_MODE='test' arriba y reejecuta.


## 9 - Inferencia sobre el TEST oficial (on-demand)

Esta seccion ejecuta inferencia sobre los **1.053 memes oficiales** y empaqueta la submission, **sin tener que cambiar `STAGE` ni reejecutar el notebook entero**. Reusa el modelo y los pools ya cargados.

Util cuando ya has hecho el sanity y solo quieres tirar el run final.


In [30]:
# === 9.1 - Inferencia JERARQUICA sobre TEST + submission =====================
# Flujo: T2.1 sobre los 1.053 memes -> T2.2/T2.3 solo sobre los YES de T2.1.
# Reutiliza el modelo, processor y POOLS ya cargados.

TEAM_NAME = "j2"   # CAMBIA esto por tu nombre de equipo
RUN_ID    = 1                   # 1, 2 o 3

# 1. Verifica paths del test
if TEST_JSON is None or not TEST_JSON.exists():
    raise RuntimeError(f"TEST_JSON no localizado: {TEST_JSON}. Revisa la celda 1.4.")
if TEST_MEME_DIR is None or not TEST_MEME_DIR.exists():
    raise RuntimeError(f"TEST_MEME_DIR no localizado: {TEST_MEME_DIR}.")

# 2. Carga test si aun no esta en df
if not (df["split"] == "test").any():
    print("Cargando TEST set en memoria...")
    with open(TEST_JSON, "r", encoding="utf-8", errors="replace") as f:
        raw_test_now = json.load(f)
    rows_test_now = _build_rows(raw_test_now, TEST_MEME_DIR, "test")
    df = pd.concat([df, pd.DataFrame(rows_test_now)], ignore_index=True)
    raw.update(raw_test_now)

df_test_now = df[df["split"] == "test"].reset_index(drop=True)
assert df_test_now["img_ok"].all(), "Hay memes test sin imagen; revisa TEST_MEME_DIR"
print(f"Test set listo: {len(df_test_now):,} memes")

test_ids = set(df_test_now["id_EXIST"].astype(str))

# Helpers locales para checkpoints de test (independientes de _CKPT_SUFFIX
# del sanity, por si el usuario corre primero sanity y luego test).
TEST_CKPT_PATHS = {t: CKPT_DIR / f"predictions_{t}_test_partial.json"
                    for t in ["t21","t22","t23"]}

def _load_test_ckpt(task):
    p = TEST_CKPT_PATHS[task]
    return json.load(open(p, "r", encoding="utf-8")) if p.exists() else {}

def _save_test_ckpt(task, preds):
    with open(TEST_CKPT_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False)

def _run_test_inference(task, target_ids):
    """Inferencia de test con checkpointing especifico."""
    eval_df = df_test_now[df_test_now["id_EXIST"].isin(target_ids)].copy().reset_index(drop=True)
    preds = _load_test_ckpt(task)
    todo = eval_df[~eval_df["id_EXIST"].isin(preds.keys())].copy()
    print()
    print(f"=== TEST {task.upper()} ===")
    print(f"  a predecir   : {len(eval_df):,}")
    print(f"  ya predichos : {len(preds):,}")
    print(f"  pendientes   : {len(todo):,}")
    if len(todo) == 0: return preds
    pbar = tqdm(todo.itertuples(index=False), total=len(todo), desc=f"test {task}")
    n_done = 0
    for row in pbar:
        mid = row.id_EXIST
        try:
            pred = predict_meme(mid, row.img_path, row.text_ocr, task, POOLS[task])
        except Exception as e:
            print()
            print(f"[err] {mid}: {type(e).__name__}: {e}")
            pred = ["NO"] if task == "t23" else "NO"
            _log_fallback(task, mid, f"<EXCEPTION: {e}>", str(pred))
        preds[mid] = pred
        n_done += 1
        if n_done % CKPT_EVERY == 0:
            _save_test_ckpt(task, preds)
    _save_test_ckpt(task, preds)
    return preds

# === FLUJO JERARQUICO TEST ===================================================
# 1) T2.1 sobre los 1.053
print()
print(f"--- Paso 1/3: T2.1 sobre {len(test_ids):,} memes de test ---")
test_preds_t21 = _run_test_inference("t21", test_ids)
test_sexist_ids = {mid for mid, v in test_preds_t21.items() if v == "YES"}
print()
print(f"=== T2.1 -> {len(test_sexist_ids):,} YES sexistas "
      f"({100*len(test_sexist_ids)/len(test_ids):.1f}%) ===")

# 2) T2.2 solo sobre sexistas; resto -> "NO"
print()
print(f"--- Paso 2/3: T2.2 sobre {len(test_sexist_ids):,} sexistas ---")
test_preds_t22 = _run_test_inference("t22", test_sexist_ids)
for mid in test_ids:
    if test_preds_t21.get(mid) == "NO":
        test_preds_t22[mid] = "NO"
_save_test_ckpt("t22", test_preds_t22)

# 3) T2.3 idem
print()
print(f"--- Paso 3/3: T2.3 sobre {len(test_sexist_ids):,} sexistas ---")
test_preds_t23 = _run_test_inference("t23", test_sexist_ids)
for mid in test_ids:
    if test_preds_t21.get(mid) == "NO":
        test_preds_t23[mid] = ["NO"]
_save_test_ckpt("t23", test_preds_t23)

# 4. Genera JSONs PyEvALL hard
TEST_PRED_PATHS = {
    "t21": PRED_DIR / "pred_t21_hard_test_nfe_.json",
    "t22": PRED_DIR / "pred_t22_hard_test_nfe_.json",
    "t23": PRED_DIR / "pred_t23_hard_test_nfe_.json",
}
print()
for task, preds in [("t21", test_preds_t21), ("t22", test_preds_t22), ("t23", test_preds_t23)]:
    pj = build_pyevall_hard(preds, task)
    _sanity_check_pred(pj, task)
    with open(TEST_PRED_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(pj, f, ensure_ascii=False, indent=2)
    print(f"[{task}] {len(pj):,} predicciones -> {TEST_PRED_PATHS[task].name}")

# 5. Cobertura
for task, preds in [("t21", test_preds_t21), ("t22", test_preds_t22), ("t23", test_preds_t23)]:
    miss = test_ids - set(preds.keys())
    extra = set(preds.keys()) - test_ids
    assert not miss,  f"[{task}] faltan {len(miss)} predicciones (e.g. {list(miss)[:3]})"
    assert not extra, f"[{task}] hay {len(extra)} predicciones fuera del test set"
print()
print(f"OK Cobertura test: {len(test_ids):,} memes en cada subtask.")

# 6. Distribucion de predicciones
print()
print("--- Distribucion de predicciones por subtask ---")
for task, preds in [("t21", test_preds_t21), ("t22", test_preds_t22), ("t23", test_preds_t23)]:
    if task == "t23":
        cnt = Counter()
        for v in preds.values():
            key = "NO" if v == ["NO"] else "+".join(sorted(v))
            cnt[key] += 1
    else:
        cnt = Counter(preds.values())
    print()
    print(f"[{task.upper()}]")
    total = sum(cnt.values())
    for k, v in cnt.most_common():
        print(f"    {str(k):<60}  {v:>5}  ({100*v/total:5.1f}%)")

# 7. Empaquetar submission
import shutil
SUBM_BASE = SUBM_DIR / f"exist2026_{TEAM_NAME}"
if SUBM_BASE.exists(): shutil.rmtree(SUBM_BASE)
SUBM_BASE.mkdir(parents=True, exist_ok=True)
name_map = {"t21": "task2_1", "t22": "task2_2", "t23": "task2_3"}
print()
print(f"--- Empaquetando submission en {SUBM_BASE.name}/ ---")
for task in ["t21","t22","t23"]:
    src = TEST_PRED_PATHS[task]
    dst = SUBM_BASE / f"{name_map[task]}_hard_{TEAM_NAME}_{RUN_ID}"
    dst.write_bytes(src.read_bytes())
    print(f"  {dst.name}  ({src.stat().st_size/1024:.1f} KB)")

zip_path = SUBM_DIR / f"exist2026_{TEAM_NAME}.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip",
                     root_dir=str(SUBM_DIR),
                     base_dir=f"exist2026_{TEAM_NAME}")
print()
print(f"OK Submission empaquetada: {zip_path}")
print(f"   Sube este ZIP al form: https://forms.gle/5hY91c7aBv563oZM7")
if TEAM_NAME == "TODO_TEAM_NAME":
    print()
    print("ATENCION: cambia TEAM_NAME por tu nombre real de equipo y reejecuta esta celda.")


Test set listo: 1,053 memes

--- Paso 1/3: T2.1 sobre 1,053 memes de test ---

=== TEST T21 ===
  a predecir   : 1,053
  ya predichos : 0
  pendientes   : 1,053


test t21:   0%|          | 0/1053 [00:00<?, ?it/s]


=== T2.1 -> 663 YES sexistas (63.0%) ===

--- Paso 2/3: T2.2 sobre 663 sexistas ---

=== TEST T22 ===
  a predecir   : 663
  ya predichos : 0
  pendientes   : 663


test t22:   0%|          | 0/663 [00:00<?, ?it/s]


--- Paso 3/3: T2.3 sobre 663 sexistas ---

=== TEST T23 ===
  a predecir   : 663
  ya predichos : 0
  pendientes   : 663


test t23:   0%|          | 0/663 [00:00<?, ?it/s]


[t21] 1,053 predicciones -> pred_t21_hard_test_nfe_.json
[t22] 1,053 predicciones -> pred_t22_hard_test_nfe_.json
[t23] 1,053 predicciones -> pred_t23_hard_test_nfe_.json

OK Cobertura test: 1,053 memes en cada subtask.

--- Distribucion de predicciones por subtask ---

[T21]
    YES                                                             663  ( 63.0%)
    NO                                                              390  ( 37.0%)

[T22]
    DIRECT                                                          619  ( 58.8%)
    NO                                                              390  ( 37.0%)
    JUDGEMENTAL                                                      44  (  4.2%)

[T23]
    NO                                                              390  ( 37.0%)
    STEREOTYPING-DOMINANCE                                          239  ( 22.7%)
    OBJECTIFICATION                                                 131  ( 12.4%)
    MISOGYNY-NON-SEXUAL-VIOLENCE                    